In [ ]:
import os
import torch


class Config:

    # =========================================================================
    # DATASET PATHS
    # =========================================================================

    CHEXPERT_IMAGE_ROOT = "/mnt/Internal/MedImage/unzip_chexpert_images/CheXpert-v1.0/train/"
    CHEXPERT_TRAIN_CSV  = "/mnt/Internal/MedImage/chexpert_balanced_for_training_3000_per_label_dis+demog+age.csv"
    CHEXPERT_VALID_CSV  = "/mnt/Internal/MedImage/chexpert_balanced_for_training_252_per_label_dis+demog+age.csv"

    MIMIC_IMAGE_ROOT    = "/mnt/External/Seagate/dawood/datasets/mimic-cxr/jpg"
    MIMIC_TRAIN_CSV     = "/mnt/External/Seagate/dawood/datasets/mimic-cxr/cleaned/mimic_clean_train.csv"
    MIMIC_VALID_CSV     = "/mnt/External/Seagate/dawood/datasets/mimic-cxr/cleaned/mimic_clean_valid.csv"

    NIH_IMAGE_ROOT      = "/mnt/External/Seagate/dawood/datasets/Chest_Xray_08/ChestX-Ray8 dataset/Images/images"
    NIH_CSV             = "/mnt/External/Seagate/dawood/datasets/Chest_Xray_08/ChestX-Ray8 dataset/Data_Entry_2017_v2020.csv"

    # =========================================================================
    # LABELS
    # =========================================================================

    NUM_CLASSES = 8
    LABEL_COLS  = [
        'No Finding', 'Atelectasis', 'Cardiomegaly',
        'Consolidation', 'Edema', 'Pleural Effusion',
        'Pneumonia', 'Pneumothorax'
    ]

    # =========================================================================
    # OUTPUT DIRS
    # =========================================================================

    OUTPUT_DIR     = "/home/dawood/lab2_rotaion/counterfactual_diff_uncertainty/"
    CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
    LOG_DIR        = os.path.join(OUTPUT_DIR, "logs")
    PLOT_DIR       = os.path.join(OUTPUT_DIR, "plots")
    RESULTS_DIR    = os.path.join(OUTPUT_DIR, "results")

    # =========================================================================
    # GENERAL
    # =========================================================================

    # NOTE: cuda:3 on this machine is the tiny 4GB NVIDIA DGX Display GPU,
    # not a compute A100 — that mismatch was the cause of the earlier
    # CUDA OutOfMemoryError. Pick an actual A100 index with free memory
    # (check `nvidia-smi` before running — indices 0/1/2/4 are A100s here).
    DEVICE = "cuda:1" if torch.cuda.is_available() else "cpu"
    IMAGE_SIZE = 224
    MEAN       = [0.485, 0.456, 0.406]
    STD        = [0.229, 0.224, 0.225]
    SEED       = 42

    # =========================================================================
    # DIFFUSION MODEL
    # =========================================================================

    # U-Net architecture
    DIFF_BASE_CHANNELS  = 64            # base channel width
    DIFF_CHANNEL_MULTS  = (1, 2, 4, 8)  # → [64, 128, 256, 512]
    DIFF_DROPOUT        = 0.1
    DIFF_ATTN_DEPTHS    = (2, 3)        # apply self-attention at depths 2,3

    # Noise schedule (linear)
    DIFF_T          = 1000
    DIFF_BETA_START = 1e-4
    DIFF_BETA_END   = 0.02

    # Training
    DIFF_EPOCHS     = 50
    DIFF_BATCH_SIZE = 32
    DIFF_LR         = 2e-4
    DIFF_GRAD_CLIP  = 1.0
    DIFF_SAVE_EVERY = 10               # save checkpoint every N epochs

    # DDIM sampling
    DDIM_STEPS = 50
    DDIM_ETA   = 0.0                   # 0 = deterministic

    # ---- Visualization-only multi-level correction (SDEdit) ----
    # These t* levels feed ONLY the qualitative per-factor images shown in
    # the diagnosis figure (via partial_correct / brightness_correct).
    # They do NOT feed U_domain — that comes from DiffusionModel.domain_score(),
    # which is direct denoising-error and needs no t*-to-factor mapping at all.
    T_STAR_LEVELS = {
        "brightness_contrast": 50,
        "noise_texture":       100,
        "scanner_artefact":    150,
        "global_structure":    200,
    }
    T_STAR_FULL = 200     # t* for the full-correction visualization image

    # ---- Diffusion-based domain score (DiffusionModel.domain_score) ----
    # U_domain is the model's own denoising error: how well the diffusion
    # model — trained only on CheXpert — predicts the noise added to this
    # image. High error = the image doesn't look like training data.
    DOMAIN_SCORE_TIMESTEPS = 10     # number of t-values sampled in [100,700]
    DOMAIN_SCORE_REPEATS   = 4      # repeat with fresh noise draws, average (reduces variance)

    # =========================================================================
    # CLASSIFIER
    # =========================================================================

    CLS_BACKBONE      = "densenet121"
    CLS_DROPOUT       = 0.3
    CLS_EPOCHS        = 30
    CLS_BATCH_SIZE    = 8
    CLS_LR            = 1e-4
    CLS_WEIGHT_DECAY  = 1e-5
    CLS_SAVE_EVERY    = 5
    CLS_GRAD_CLIP     = 1.0
    CLS_CKPT_NAME     = "classifier_best.pt"   # the checkpoint that actually exists on disk

    # MC Dropout uncertainty estimation
    MC_SAMPLES        = 30             # forward passes for MC dropout

    # =========================================================================
    # CD-DSD INFERENCE
    # =========================================================================

    # Max samples to diagnose per dataset during evaluation
    MAX_DIAG_SAMPLES  = 200

    # Threshold: images with U_total below this are considered certain
    UNCERTAINTY_THR   = 0.1

### Datasets

In [ ]:
import os
import logging
import warnings

import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(message)s')
logger = logging.getLogger(__name__)


# ============================================================================
# TRANSFORMS
# ============================================================================

def get_train_transform(image_size, mean, std):
    return transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=10),
        transforms.ColorJitter(brightness=0.1, contrast=0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std),
    ])


def get_eval_transform(image_size, mean, std):
    return transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std),
    ])


def get_diffusion_transform(image_size, mean=None, std=None):
    """
    Transform for diffusion model training.

    IMPORTANT: We use the same ImageNet normalization as the classifier
    (mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]) rather than [-1,1].
    This ensures the SAME preprocessed image can be used for both diffusion
    scoring and classification at test time — no dual-transform bookkeeping.

    The U-Net handles this range fine; the key is consistency between
    training and inference normalization.
    """
    if mean is None:
        mean = [0.485, 0.456, 0.406]
    if std is None:
        std = [0.229, 0.224, 0.225]
    return transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std),
    ])


# ============================================================================
# BASE DATASET
# ============================================================================

class BaseXRayDataset(Dataset):
    """
    Abstract base class for all X-ray datasets.
    Handles validation, fallback, and common label extraction logic.
    """

    def __init__(self, label_cols, transform=None):
        self.label_cols = label_cols
        self.transform = transform
        self.dataframe = None      # Subclass must set this before calling _validate
        self.valid_indices = []

    def _validate_dataset(self, path_fn):
        """
        Validate that image files exist and are readable.
        path_fn: callable that takes a DataFrame row and returns image path string.
        """
        logger.info(f"Validating {self.__class__.__name__}...")
        for idx in range(len(self.dataframe)):
            row = self.dataframe.iloc[idx]
            path = path_fn(row)
            if os.path.exists(path):
                try:
                    with Image.open(path) as img:
                        img.verify()
                    self.valid_indices.append(idx)
                except Exception:
                    pass

        total = len(self.dataframe)
        valid = len(self.valid_indices)
        logger.info(f"  Valid: {valid}/{total} images ({100*valid/total:.1f}%)")

        if valid == 0:
            raise ValueError(
                f"No valid images found for {self.__class__.__name__}. "
                "Check your image root paths in config.py."
            )

    def _load_image(self, path):
        image = Image.open(path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image

    def _get_label(self, row):
        label = row[self.label_cols].values.astype(np.float32)
        label = np.clip(label, 0, 1)   # ensure binary
        return torch.tensor(label)

    def _fallback(self):
        h = w = 224
        if self.transform is not None:
            # Try to infer size from transform
            for t in self.transform.transforms:
                if hasattr(t, 'size'):
                    h = w = t.size if isinstance(t.size, int) else t.size[0]
                    break
        return torch.zeros(3, h, w), torch.zeros(len(self.label_cols)), -1

    def __len__(self):
        return len(self.valid_indices)


# ============================================================================
# CHEXPERT DATASET  (Source domain)
# ============================================================================

class CheXpertDataset(BaseXRayDataset):
    """
    CheXpert dataset — used as source domain.
    Splits:
        - Training   : CHEXPERT_TRAIN_CSV  (Stage 1 diffusion + Stage 2 classifier)
        - Validation : CHEXPERT_VALID_CSV  (split 50/50 → calibration + evaluation)
    """

    def __init__(self, csv_path, image_root, label_cols, transform=None):
        super().__init__(label_cols, transform)
        self.image_root = image_root
        self.dataframe = pd.read_csv(csv_path)

        # Fill NaN labels with 0, clip negatives
        for col in self.label_cols:
            if col in self.dataframe.columns:
                self.dataframe[col] = self.dataframe[col].fillna(0).clip(lower=0)
            else:
                self.dataframe[col] = 0.0

        self._validate_dataset(self._path_fn)

    def _path_fn(self, row):
        raw = row.get('Path', row.get('path', ''))
        # CheXpert paths often start with "CheXpert-v1.0/train/" — strip prefix
        relative = str(raw).replace("CheXpert-v1.0/train/", "")
        return os.path.join(self.image_root, relative)

    def __getitem__(self, idx):
        actual_idx = self.valid_indices[idx]
        row = self.dataframe.iloc[actual_idx]
        path = self._path_fn(row)
        try:
            image = self._load_image(path)
            label = self._get_label(row)
            return image, label, actual_idx
        except Exception as e:
            logger.warning(f"Error loading {path}: {e}")
            return self._fallback()


# ============================================================================
# MIMIC-CXR DATASET  (Target domain 1)
# ============================================================================

class MIMICCXRDataset(BaseXRayDataset):
    """
    MIMIC-CXR dataset — Target domain 1.
    Expected CSV columns: subject_id, study_id, dicom_id, + label cols.
    OR a 'path'/'Path' column with relative paths.
    """

    def __init__(self, csv_path, image_root, label_cols, transform=None, max_samples=None):
        super().__init__(label_cols, transform)
        self.image_root = image_root
        self.dataframe = pd.read_csv(csv_path)

        if max_samples is not None:
            self.dataframe = self.dataframe.head(max_samples)

        for col in self.label_cols:
            if col in self.dataframe.columns:
                self.dataframe[col] = self.dataframe[col].fillna(0).clip(lower=0)
            else:
                self.dataframe[col] = 0.0

        self._validate_dataset(self._path_fn)

    def _path_fn(self, row):
        if 'path' in row:
            return os.path.join(self.image_root, str(row['path']))
        if 'Path' in row:
            return os.path.join(self.image_root, str(row['Path']))
        # Construct from MIMIC hierarchical structure
        subj = str(row['subject_id'])
        subject_folder = f"p{subj[:2]}/p{subj}"
        study_folder   = f"s{row['study_id']}"
        dicom_file     = f"{row['dicom_id']}.jpg"
        return os.path.join(self.image_root, subject_folder, study_folder, dicom_file)

    def __getitem__(self, idx):
        actual_idx = self.valid_indices[idx]
        row = self.dataframe.iloc[actual_idx]
        path = self._path_fn(row)
        try:
            image = self._load_image(path)
            label = self._get_label(row)
            return image, label, actual_idx
        except Exception as e:
            logger.warning(f"Error loading {path}: {e}")
            return self._fallback()


# ============================================================================
# NIH CHESTX-RAY14 DATASET  (Target domain 2)
# ============================================================================

class NIHChestXrayDataset(BaseXRayDataset):
    """
    NIH ChestX-ray14 dataset — Target domain 2.
    Remaps NIH label names to CheXpert 8-label format.
    """

    # NIH label name  →  CheXpert label name
    NIH_TO_CHEXPERT = {
        'Atelectasis':  'Atelectasis',
        'Cardiomegaly': 'Cardiomegaly',
        'Consolidation':'Consolidation',
        'Edema':        'Edema',
        'Effusion':     'Pleural Effusion',   # NIH uses "Effusion"
        'Pneumonia':    'Pneumonia',
        'Pneumothorax': 'Pneumothorax',
    }

    def __init__(self, csv_path, image_root, label_cols, transform=None, max_samples=None):
        super().__init__(label_cols, transform)
        self.image_root = image_root
        self.dataframe = pd.read_csv(csv_path)
        
        if max_samples is not None:          # add these two lines
            self.dataframe = self.dataframe.head(max_samples)
        
        self._remap_labels()
        self._validate_dataset(self._path_fn)

    def _remap_labels(self):
        """Convert NIH multi-label string to binary columns matching LABEL_COLS."""
        for col in self.label_cols:
            self.dataframe[col] = 0.0

        for idx, row in self.dataframe.iterrows():
            findings = str(row.get('Finding Labels', '')).split('|')
            findings = [f.strip() for f in findings]

            if 'No Finding' in findings:
                self.dataframe.at[idx, 'No Finding'] = 1.0
            else:
                for nih_label, chex_label in self.NIH_TO_CHEXPERT.items():
                    if nih_label in findings and chex_label in self.label_cols:
                        self.dataframe.at[idx, chex_label] = 1.0

    def _path_fn(self, row):
        return os.path.join(self.image_root, str(row['Image Index']))

    def __getitem__(self, idx):
        actual_idx = self.valid_indices[idx]
        row = self.dataframe.iloc[actual_idx]
        path = self._path_fn(row)
        try:
            image = self._load_image(path)
            label = self._get_label(row)
            return image, label, actual_idx
        except Exception as e:
            logger.warning(f"Error loading {path}: {e}")
            return self._fallback()


# ============================================================================
# COLLATE FUNCTION
# ============================================================================

def collate_fn(batch):
    """Robust collate — filters out broken/fallback samples."""
    valid = [item for item in batch if item is not None and item[2] != -1]
    if len(valid) == 0:
        n = len(batch)
        return torch.zeros(n, 3, 224, 224), torch.zeros(n, 8), [-1] * n

    images, labels, indices = zip(*valid)
    return (
        torch.stack([torch.as_tensor(img) for img in images]),
        torch.stack([torch.as_tensor(lbl) for lbl in labels]),
        list(indices)
    )


# ============================================================================
# DATALOADER FACTORY
# ============================================================================

def get_dataloader(dataset, batch_size, shuffle=True, num_workers=4, pin_memory=True):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        collate_fn=collate_fn,
        pin_memory=pin_memory,
        drop_last=False,
    )


### UNET

In [ ]:
"""
U-Net backbone for the DDPM diffusion model.

Architecture (with base_channels=64, channel_mults=(1,2,4,8), image 224x224):

  init_conv :  3 → 64   (224)
  enc0      : 64 → 64   (224 → 112)
  enc1      : 64 → 128  (112 → 56)
  enc2      : 128 → 256 (56  → 28)   ← self-attention
  enc3      : 256 → 512 (28  → 14)   ← self-attention
  mid       : 512 → 512 (14)         ← self-attention
  dec0      : 512+512→256 (14 → 28)  ← self-attention
  dec1      : 256+256→128 (28 → 56)  ← self-attention
  dec2      : 128+128→64  (56 → 112)
  dec3      :  64+64→ 64  (112 → 224)
  final_conv:  64 → 3    (224)
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


# ---------------------------------------------------------------------------
# Utilities
# ---------------------------------------------------------------------------

def num_groups(channels: int, max_groups: int = 32) -> int:
    """Return largest divisor of `channels` that is <= max_groups."""
    g = max_groups
    while g > 1 and channels % g != 0:
        g -= 1
    return g


def sinusoidal_embedding(timesteps: torch.Tensor, dim: int,
                         max_period: int = 10000) -> torch.Tensor:
    """Classic sinusoidal positional embedding for timesteps."""
    assert dim % 2 == 0
    half = dim // 2
    freqs = torch.exp(
        -math.log(max_period)
        * torch.arange(half, dtype=torch.float32, device=timesteps.device)
        / half
    )
    args = timesteps[:, None].float() * freqs[None]
    return torch.cat([torch.cos(args), torch.sin(args)], dim=-1)  # (B, dim)


# ---------------------------------------------------------------------------
# Core blocks
# ---------------------------------------------------------------------------

class TimeEmbedding(nn.Module):
    """Sinusoidal embedding → 2-layer MLP → out_dim."""

    def __init__(self, base_dim: int, out_dim: int):
        super().__init__()
        self.base_dim = base_dim
        self.mlp = nn.Sequential(
            nn.Linear(base_dim, out_dim),
            nn.SiLU(),
            nn.Linear(out_dim, out_dim),
        )

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        return self.mlp(sinusoidal_embedding(t, self.base_dim))


class ResBlock(nn.Module):
    """
    ResNet-style block conditioned on timestep embedding.
      x → GroupNorm → SiLU → Conv → + time_proj(t_emb) → GroupNorm → SiLU → Dropout → Conv → + shortcut
    """

    def __init__(self, in_ch: int, out_ch: int, t_dim: int, dropout: float = 0.1):
        super().__init__()
        self.norm1 = nn.GroupNorm(num_groups(in_ch), in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)

        self.t_proj = nn.Sequential(
            nn.SiLU(),
            nn.Linear(t_dim, out_ch),
        )

        self.norm2   = nn.GroupNorm(num_groups(out_ch), out_ch)
        self.drop    = nn.Dropout(dropout)
        self.conv2   = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.shortcut = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x: torch.Tensor, t_emb: torch.Tensor) -> torch.Tensor:
        h = self.conv1(F.silu(self.norm1(x)))
        h = h + self.t_proj(t_emb)[:, :, None, None]   # broadcast over H, W
        h = self.conv2(self.drop(F.silu(self.norm2(h))))
        return h + self.shortcut(x)


class AttentionBlock(nn.Module):
    """
    Multi-head self-attention for spatial feature maps.
    Reshape (B,C,H,W) → (B, H*W, C), attend, reshape back.
    """

    def __init__(self, channels: int, num_heads: int = 8):
        super().__init__()
        # Ensure num_heads divides channels
        while channels % num_heads != 0 and num_heads > 1:
            num_heads //= 2
        self.norm = nn.GroupNorm(num_groups(channels), channels)
        self.attn = nn.MultiheadAttention(channels, num_heads, batch_first=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, H, W = x.shape
        h = self.norm(x).reshape(B, C, H * W).permute(0, 2, 1)  # (B, HW, C)
        h, _ = self.attn(h, h, h, need_weights=False)
        return x + h.permute(0, 2, 1).reshape(B, C, H, W)


# ---------------------------------------------------------------------------
# Encoder / Decoder building blocks
# ---------------------------------------------------------------------------

class EncoderBlock(nn.Module):
    """ResBlock (+ optional attention) → skip → strided-conv downsample."""

    def __init__(self, in_ch: int, out_ch: int, t_dim: int,
                 has_attn: bool = False, dropout: float = 0.1):
        super().__init__()
        self.res  = ResBlock(in_ch, out_ch, t_dim, dropout)
        self.attn = AttentionBlock(out_ch) if has_attn else nn.Identity()
        self.down = nn.Conv2d(out_ch, out_ch, 4, stride=2, padding=1)

    def forward(self, x, t_emb):
        x    = self.res(x, t_emb)
        x    = self.attn(x)
        skip = x                           # save before downsampling
        x    = self.down(x)
        return x, skip


class DecoderBlock(nn.Module):
    """ConvTranspose upsample → concat skip → ResBlock (+ optional attention)."""

    def __init__(self, in_ch: int, skip_ch: int, out_ch: int, t_dim: int,
                 has_attn: bool = False, dropout: float = 0.1):
        super().__init__()
        self.up   = nn.ConvTranspose2d(in_ch, in_ch, 4, stride=2, padding=1)
        self.res  = ResBlock(in_ch + skip_ch, out_ch, t_dim, dropout)
        self.attn = AttentionBlock(out_ch) if has_attn else nn.Identity()

    def forward(self, x, skip, t_emb):
        x = self.up(x)
        x = torch.cat([x, skip], dim=1)
        x = self.res(x, t_emb)
        x = self.attn(x)
        return x


# ---------------------------------------------------------------------------
# Full U-Net
# ---------------------------------------------------------------------------

class UNet(nn.Module):
    """
    Time-conditioned U-Net for DDPM noise prediction.

    Parameters
    ----------
    in_channels   : image channels (3 for RGB)
    base_channels : channel width at level 0
    channel_mults : channel multipliers at each encoder depth
    t_dim_base    : sinusoidal embedding base dimension (projected to base_channels * 4)
    dropout       : dropout rate inside ResBlocks
    attn_depths   : tuple of encoder depth indices where attention is applied
    """

    def __init__(
        self,
        in_channels:   int   = 3,
        base_channels: int   = 64,
        channel_mults: tuple = (1, 2, 4, 8),
        t_dim_base:    int   = 256,
        dropout:       float = 0.1,
        attn_depths:   tuple = (2, 3),
    ):
        super().__init__()

        t_dim   = base_channels * 4
        chs     = [base_channels * m for m in channel_mults]   # e.g. [64,128,256,512]
        n_lvls  = len(chs)

        # Time embedding
        self.time_embed = TimeEmbedding(t_dim_base, t_dim)

        # Initial projection
        self.init_conv = nn.Conv2d(in_channels, base_channels, 3, padding=1)

        # ---- Encoder ----
        self.enc_blocks = nn.ModuleList()
        in_ch = base_channels
        for depth, out_ch in enumerate(chs):
            has_attn = depth in attn_depths
            self.enc_blocks.append(EncoderBlock(in_ch, out_ch, t_dim, has_attn, dropout))
            in_ch = out_ch

        # ---- Middle ----
        mid_ch = chs[-1]
        self.mid_res1 = ResBlock(mid_ch, mid_ch, t_dim, dropout)
        self.mid_attn = AttentionBlock(mid_ch)
        self.mid_res2 = ResBlock(mid_ch, mid_ch, t_dim, dropout)

        # ---- Decoder ----
        # dec[i] receives:  in_ch (from previous), skip from enc[n_lvls-1-i]
        # dec[i] outputs:   chs[n_lvls-2-i]  (or base_channels for last)
        self.dec_blocks = nn.ModuleList()
        for i in range(n_lvls):
            skip_ch = chs[n_lvls - 1 - i]
            out_idx = n_lvls - 2 - i
            out_ch  = chs[out_idx] if out_idx >= 0 else base_channels
            has_attn = (n_lvls - 1 - i) in attn_depths
            self.dec_blocks.append(DecoderBlock(in_ch, skip_ch, out_ch, t_dim, has_attn, dropout))
            in_ch = out_ch

        # ---- Final projection ----
        self.final_conv = nn.Sequential(
            nn.GroupNorm(num_groups(in_ch), in_ch),
            nn.SiLU(),
            nn.Conv2d(in_ch, in_channels, 1),
        )

    # ------------------------------------------------------------------

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        """
        Parameters
        ----------
        x : (B, C, H, W)  noisy image
        t : (B,)           integer timesteps

        Returns
        -------
        eps_pred : (B, C, H, W)  predicted noise
        """
        t_emb = self.time_embed(t)          # (B, t_dim)

        x = self.init_conv(x)               # (B, base_ch, H, W)

        # Encoder — collect skips
        skips = []
        for enc in self.enc_blocks:
            x, skip = enc(x, t_emb)
            skips.append(skip)

        # Middle
        x = self.mid_res1(x, t_emb)
        x = self.mid_attn(x)
        x = self.mid_res2(x, t_emb)

        # Decoder — consume skips in reverse
        for dec, skip in zip(self.dec_blocks, reversed(skips)):
            x = dec(x, skip, t_emb)

        return self.final_conv(x)

### Diffusion

In [ ]:
"""
DDPM noise schedule + DDIM sampler.

Key functions used by CD-DSD:
  - DiffusionModel.train_step()       : standard DDPM loss
  - DiffusionModel.domain_score()     : denoising-error domain-shift score
                                         used directly as U_domain in CD-DSD
  - DiffusionModel.partial_correct()  : SDEdit-style correction, used ONLY
                                         for the illustrative visualization
                                         panel (brightness/noise/scanner/
                                         structure images) — not part of
                                         any reported score
  - DiffusionModel.ddim_sample()      : full DDIM denoising from noise
  - DiffusionModel.ddim_invert()      : deterministic encode x0 → x_T,
                                         kept as a utility, currently unused
                                         by CD-DSD
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from tqdm import tqdm


# ---------------------------------------------------------------------------
# Noise schedule helpers
# ---------------------------------------------------------------------------

def linear_beta_schedule(T: int, beta_start: float, beta_end: float) -> torch.Tensor:
    return torch.linspace(beta_start, beta_end, T)


def cosine_beta_schedule(T: int, s: float = 0.008) -> torch.Tensor:
    steps = T + 1
    t = torch.linspace(0, T, steps) / T
    f = torch.cos((t + s) / (1 + s) * np.pi / 2) ** 2
    alphas_cumprod = f / f[0]
    betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
    return betas.clamp(1e-4, 0.9999)


def precompute_schedule(betas: torch.Tensor) -> dict:
    alphas           = 1.0 - betas
    alpha_bar        = torch.cumprod(alphas, dim=0)
    alpha_bar_prev   = F.pad(alpha_bar[:-1], (1, 0), value=1.0)

    sqrt_ab          = alpha_bar.sqrt()
    sqrt_one_m_ab    = (1.0 - alpha_bar).sqrt()
    sqrt_recip_ab    = (1.0 / alpha_bar).sqrt()
    sqrt_recip_m1_ab = (1.0 / alpha_bar - 1.0).sqrt()

    post_var         = betas * (1 - alpha_bar_prev) / (1 - alpha_bar)
    post_mean_c1     = betas * alpha_bar_prev.sqrt() / (1 - alpha_bar)
    post_mean_c2     = (1 - alpha_bar_prev) * alphas.sqrt() / (1 - alpha_bar)

    return dict(
        betas            = betas,
        alphas           = alphas,
        alpha_bar        = alpha_bar,
        alpha_bar_prev   = alpha_bar_prev,
        sqrt_ab          = sqrt_ab,
        sqrt_one_m_ab    = sqrt_one_m_ab,
        sqrt_recip_ab    = sqrt_recip_ab,
        sqrt_recip_m1_ab = sqrt_recip_m1_ab,
        post_var         = post_var,
        post_mean_c1     = post_mean_c1,
        post_mean_c2     = post_mean_c2,
    )


# ---------------------------------------------------------------------------
# DiffusionModel
# ---------------------------------------------------------------------------

class DiffusionModel(nn.Module):

    def __init__(
        self,
        unet,
        T:          int   = 1000,
        beta_start: float = 1e-4,
        beta_end:   float = 0.02,
        schedule:   str   = "linear",
    ):
        super().__init__()
        self.unet = unet
        self.T    = T

        if schedule == "cosine":
            betas = cosine_beta_schedule(T)
        else:
            betas = linear_beta_schedule(T, beta_start, beta_end)

        sched = precompute_schedule(betas)
        for k, v in sched.items():
            self.register_buffer(k, v.float())

    # ------------------------------------------------------------------
    # Training loss
    # ------------------------------------------------------------------

    def train_step(self, x0: torch.Tensor) -> torch.Tensor:
        B      = x0.shape[0]
        device = x0.device
        t      = torch.randint(0, self.T, (B,), device=device)
        eps    = torch.randn_like(x0)
        x_t    = (
            self.sqrt_ab[t, None, None, None] * x0
            + self.sqrt_one_m_ab[t, None, None, None] * eps
        )
        eps_pred = self.unet(x_t, t)
        return F.mse_loss(eps_pred, eps)

    # ------------------------------------------------------------------
    # Internal: single DDIM reverse step  x_t → x_{t_prev}
    # ------------------------------------------------------------------

    @torch.no_grad()
    def _ddim_step(self, x_t, t, t_prev, eta=0.0):
        device   = x_t.device
        t_batch  = torch.full((x_t.shape[0],), t, device=device, dtype=torch.long)

        eps_pred = self.unet(x_t, t_batch)
        ab_t     = self.alpha_bar[t]
        ab_tp    = self.alpha_bar[t_prev] if t_prev >= 0 else torch.tensor(1.0, device=device)

        pred_x0  = (x_t - self.sqrt_one_m_ab[t] * eps_pred) / self.sqrt_ab[t]
        pred_x0  = pred_x0.clamp(-4, 4)

        sigma_t  = eta * ((1 - ab_tp) / (1 - ab_t) * (1 - ab_t / ab_tp)).sqrt()
        dir_xt   = (1 - ab_tp - sigma_t ** 2).sqrt() * eps_pred
        noise    = sigma_t * torch.randn_like(x_t) if eta > 0 else 0

        return ab_tp.sqrt() * pred_x0 + dir_xt + noise

    # ------------------------------------------------------------------
    # Internal: single DDIM FORWARD step  x_t → x_{t_next}  (inversion)
    # ------------------------------------------------------------------

    @torch.no_grad()
    def _ddim_invert_step(self, x_t, t, t_next):
        """
        One deterministic forward step used during DDIM inversion.
        Goes x_t → x_{t_next} where t_next > t (moving toward more noise).
        """
        device  = x_t.device
        t_batch = torch.full((x_t.shape[0],), t, device=device, dtype=torch.long)

        eps_pred = self.unet(x_t, t_batch)
        ab_t     = self.alpha_bar[t]
        ab_next  = self.alpha_bar[t_next]

        # DDIM inversion formula (deterministic, eta=0)
        pred_x0 = (x_t - (1 - ab_t).sqrt() * eps_pred) / ab_t.sqrt()
        pred_x0 = pred_x0.clamp(-4, 4)

        x_next  = ab_next.sqrt() * pred_x0 + (1 - ab_next).sqrt() * eps_pred
        return x_next

    # ------------------------------------------------------------------
    # Full DDIM sampling  noise → image
    # ------------------------------------------------------------------

    @torch.no_grad()
    def ddim_sample(self, x_T, num_steps=50, eta=0.0, verbose=False):
        timesteps = torch.linspace(self.T - 1, 0, num_steps + 1).long().tolist()
        x     = x_T
        pairs = list(zip(timesteps[:-1], timesteps[1:]))
        for t, t_prev in (tqdm(pairs, desc="DDIM sample") if verbose else pairs):
            x = self._ddim_step(x, int(t), int(t_prev), eta)
        return x

    # ------------------------------------------------------------------
    # DDIM Inversion  x0 → x_T  (deterministic encoding)
    # ------------------------------------------------------------------

    @torch.no_grad()
    def ddim_invert(self, x0, num_steps=50):
        """
        Deterministically encode x0 → x_T using DDIM inversion.

        In-domain images (CheXpert) encode cleanly because the diffusion
        model knows their distribution well.
        Out-of-domain images (MIMIC, NIH) encode poorly — the model's
        noise predictions are inaccurate, so the encoding drifts.
        This drift is what makes the reconstruction distance meaningful.
        """
        timesteps = torch.linspace(0, self.T - 1, num_steps + 1).long().tolist()
        x = x0
        for t, t_next in zip(timesteps[:-1], timesteps[1:]):
            x = self._ddim_invert_step(x, int(t), int(t_next))
        return x   # x_T

    # ------------------------------------------------------------------
    # Invert + Reconstruct  ← Core domain correction for CD-DSD
    # ------------------------------------------------------------------

    @torch.no_grad()
    def domain_score(self, x0, n_timesteps=10):
        """
        Compute domain shift score as mean noise prediction error
        across multiple timesteps.
        
        Low score  → image looks like training data (CheXpert)
        High score → image is out-of-domain (MIMIC, NIH)
        """
        device = x0.device
        B      = x0.shape[0]
        errors = []

        # Sample n_timesteps evenly spread between t=100 and t=700
        # (avoid very low t where all images look similar,
        #  avoid very high t where everything is pure noise)
        timesteps = torch.linspace(100, 700, n_timesteps).long().to(device)

        for t_val in timesteps:
            t_batch = t_val.expand(B)
            eps     = torch.randn_like(x0)

            # Corrupt image to timestep t
            x_t = (
                self.sqrt_ab[t_val] * x0
                + self.sqrt_one_m_ab[t_val] * eps
            )

            # Model tries to predict the noise
            eps_pred = self.unet(x_t, t_batch)

            # MSE between true noise and predicted noise, per image
            mse = (eps_pred - eps).pow(2).mean(dim=(1, 2, 3))  # (B,)
            errors.append(mse)

        # Average across all timesteps → single score per image
        return torch.stack(errors).mean(dim=0)   # (B,)

    @torch.no_grad()
    def brightness_correct(self, x):
        """
        Simple intensity normalisation — shift mean and std 
        back to training distribution before diffusion correction.
        This handles brightness and contrast shifts explicitly.
        """
        # Per-image normalisation to zero mean unit std
        mean = x.mean(dim=(1,2,3), keepdim=True)
        std  = x.std(dim=(1,2,3),  keepdim=True).clamp(min=1e-6)
        x_norm = (x - mean) / std
        
        # Then scale to match training distribution statistics
        # Use CheXpert training mean/std (your cfg.MEAN, cfg.STD)
        target_mean = torch.tensor(0.0, device=x.device)  
        target_std  = torch.tensor(1.0, device=x.device)
        return x_norm * target_std + target_mean
    # ------------------------------------------------------------------
    # SDEdit-style partial correction  ← used for factor attribution only
    # ------------------------------------------------------------------

    @torch.no_grad()
    def partial_correct(self, x_test, t_star, num_steps=50, eta=0.0):
        """
        SDEdit partial correction at noise level t_star.
        Used ONLY for factor attribution (brightness/noise/scanner/structure).
        NOT used for U_domain computation (use ddim_invert_and_reconstruct instead).

        Lower t_star → corrects only low-level factors (brightness, contrast)
        Higher t_star → corrects deeper factors (scanner artefacts, structure)
        """
        device   = x_test.device
        noise    = torch.randn_like(x_test)
        ab_t     = self.alpha_bar[t_star]
        x_noised = ab_t.sqrt() * x_test + (1 - ab_t).sqrt() * noise

        timesteps = torch.linspace(t_star, 0, num_steps + 1).long().clamp(0, self.T - 1).tolist()
        x = x_noised
        for t, t_prev in zip(timesteps[:-1], timesteps[1:]):
            x = self._ddim_step(x, int(t), int(t_prev), eta)
        return x

    # ------------------------------------------------------------------
    # Reconstruction score (auxiliary OOD signal)
    # ------------------------------------------------------------------

    @torch.no_grad()
    def reconstruction_score(self, x_test, t_star, n_samples=5):
        errors = []
        for _ in range(n_samples):
            x_star = self.partial_correct(x_test, t_star, num_steps=50, eta=0.2)
            errors.append(F.mse_loss(x_star, x_test, reduction='none').mean(dim=(1, 2, 3)))
        return torch.stack(errors).mean(dim=0)

### Classifier

In [ ]:
"""
Multi-label chest X-ray classifier with MC-Dropout uncertainty estimation.

Backbone : DenseNet-121 (ImageNet pretrained)
Head     : Linear(1024 → num_classes) with dropout
Uncertainty : MC Dropout — run N stochastic forward passes and measure
              predictive entropy / variance across samples.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models


# ---------------------------------------------------------------------------
# Helper: enable dropout during inference
# ---------------------------------------------------------------------------

def enable_dropout(model: nn.Module) -> None:
    """Set all Dropout layers to train mode (so they drop during inference)."""
    for m in model.modules():
        if isinstance(m, (nn.Dropout, nn.Dropout2d)):
            m.train()


# ---------------------------------------------------------------------------
# Classifier
# ---------------------------------------------------------------------------

class MCDropoutClassifier(nn.Module):
    """
    DenseNet-121 with an MC-Dropout head for multi-label classification.

    The dropout layer sits between the global pooling output and the
    final linear projection.  At inference time, calling `enable_dropout()`
    before forward passes activates stochastic sampling.
    """

    def __init__(self, num_classes: int = 8, dropout_rate: float = 0.3,
                 pretrained: bool = True):
        super().__init__()

        backbone = models.densenet121(
            weights=models.DenseNet121_Weights.IMAGENET1K_V1 if pretrained else None
        )

        # Everything except the original classifier
        self.features = backbone.features
        self.pool     = nn.AdaptiveAvgPool2d(1)
        self.drop     = nn.Dropout(p=dropout_rate)
        self.fc       = nn.Linear(1024, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Returns raw logits (B, num_classes).
        Apply sigmoid externally for probabilities.
        """
        f = self.features(x)
        f = F.relu(f, inplace=True)
        f = self.pool(f).flatten(1)       # (B, 1024)
        f = self.drop(f)
        return self.fc(f)                 # (B, num_classes)

    # ------------------------------------------------------------------
    # Uncertainty estimation
    # ------------------------------------------------------------------

    @torch.no_grad()
    def mc_predict(
        self,
        x:         torch.Tensor,
        n_samples: int = 30,
    ) -> dict:
        """
        Run `n_samples` stochastic forward passes with MC Dropout.

        Returns
        -------
        dict with keys:
          mean_prob  : (B, C)  — mean predicted probability per class
          variance   : (B, C)  — variance across MC samples
          entropy    : (B,)    — predictive entropy (uncertainty scalar)
          raw_probs  : (B, n_samples, C)  — all sampled probabilities
        """
        self.eval()
        enable_dropout(self)   # activate dropout during inference

        probs_list = []
        for _ in range(n_samples):
            logits = self(x)
            probs_list.append(torch.sigmoid(logits))   # (B, C)

        probs = torch.stack(probs_list, dim=1)         # (B, n_samples, C)

        mean_prob = probs.mean(dim=1)                  # (B, C)
        variance  = probs.var(dim=1)                   # (B, C)

        # Predictive entropy over mean probabilities (multi-label)
        eps = 1e-7
        p   = mean_prob.clamp(eps, 1 - eps)
        entropy_per_class = -(p * p.log() + (1 - p) * (1 - p).log())
        entropy = entropy_per_class.mean(dim=1)        # (B,) — average over classes

        return dict(
            mean_prob  = mean_prob,
            variance   = variance,
            entropy    = entropy,
            raw_probs  = probs,
        )

    @torch.no_grad()
    def uncertainty_scalar(
        self,
        x:         torch.Tensor,
        n_samples: int = 30,
    ) -> torch.Tensor:
        """
        Convenience method: returns a single uncertainty scalar per image.
        Shape: (B,)
        """
        return self.mc_predict(x, n_samples)["entropy"]

    # ------------------------------------------------------------------
    # Standard deterministic inference (no dropout)
    # ------------------------------------------------------------------

    @torch.no_grad()
    def predict(self, x: torch.Tensor) -> torch.Tensor:
        """Deterministic prediction; returns sigmoid probabilities (B, C)."""
        self.eval()
        return torch.sigmoid(self(x))

### Train_Classifier

In [ ]:
"""
Stage 1: Train the multi-label DenseNet-121 classifier on CheXpert.

Run:
    python train_classifier.py
"""

import os
import logging
import time
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import roc_auc_score
import numpy as np
from tqdm import tqdm

# ---------------------------------------------------------------------------
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger(__name__)
# ---------------------------------------------------------------------------


def compute_class_weights(dataset) -> torch.Tensor:
    """Positive/negative ratio per class to counter class imbalance."""
    labels = []
    for idx in dataset.valid_indices:
        row  = dataset.dataframe.iloc[idx]
        lab  = row[dataset.label_cols].values.astype(float)
        labels.append(lab)
    labels = np.array(labels)                             # (N, C)

    pos   = labels.sum(0).clip(1)
    neg   = (1 - labels).sum(0).clip(1)
    w_pos = neg / pos                                     # larger weight for rare positives
    return torch.tensor(w_pos, dtype=torch.float32)


def cls_train_one_epoch(model, loader, optimizer, criterion, device, grad_clip):
    model.train()
    total_loss, n = 0.0, 0
    for imgs, labels, _ in tqdm(loader, desc="  train", leave=False):
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()
        logits = model(imgs)
        loss   = criterion(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()

        total_loss += loss.item() * imgs.size(0)
        n          += imgs.size(0)

    return total_loss / n


@torch.no_grad()
def evaluate(model, loader, criterion, device, label_cols):
    model.eval()
    total_loss, n = 0.0, 0
    all_preds, all_labels = [], []

    for imgs, labels, _ in tqdm(loader, desc="  eval ", leave=False):
        imgs, labels = imgs.to(device), labels.to(device)
        logits       = model(imgs)
        loss         = criterion(logits, labels)

        total_loss  += loss.item() * imgs.size(0)
        n           += imgs.size(0)
        all_preds.append(torch.sigmoid(logits).cpu().numpy())
        all_labels.append(labels.cpu().numpy())

    preds  = np.concatenate(all_preds,  axis=0)
    labs   = np.concatenate(all_labels, axis=0)
    avg_loss = total_loss / n

    # Per-class AUC (skip classes with only one label present)
    aucs = []
    for c in range(len(label_cols)):
        if labs[:, c].sum() > 0 and (1 - labs[:, c]).sum() > 0:
            try:
                aucs.append(roc_auc_score(labs[:, c], preds[:, c]))
            except Exception:
                pass

    mean_auc = float(np.mean(aucs)) if aucs else 0.0
    return avg_loss, mean_auc


# ---------------------------------------------------------------------------

def train_classifier_main():
    cfg    = Config()
    device = torch.device(cfg.DEVICE)

    # Reproducibility
    torch.manual_seed(cfg.SEED)
    np.random.seed(cfg.SEED)

    # Create output dirs
    Path(cfg.CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)
    Path(cfg.LOG_DIR).mkdir(parents=True, exist_ok=True)

    # ---- Datasets ----
    logger.info("Loading datasets...")
    train_tf = get_train_transform(cfg.IMAGE_SIZE, cfg.MEAN, cfg.STD)
    eval_tf  = get_eval_transform(cfg.IMAGE_SIZE, cfg.MEAN, cfg.STD)

    train_ds = CheXpertDataset(
        csv_path   = cfg.CHEXPERT_TRAIN_CSV,
        image_root = cfg.CHEXPERT_IMAGE_ROOT,
        label_cols = cfg.LABEL_COLS,
        transform  = train_tf,
    )
    valid_ds = CheXpertDataset(
        csv_path   = cfg.CHEXPERT_VALID_CSV,
        image_root = cfg.CHEXPERT_IMAGE_ROOT,
        label_cols = cfg.LABEL_COLS,
        transform  = eval_tf,
    )

    train_loader = get_dataloader(train_ds, cfg.CLS_BATCH_SIZE, shuffle=True)
    valid_loader = get_dataloader(valid_ds, cfg.CLS_BATCH_SIZE, shuffle=False)
    logger.info(f"Train: {len(train_ds):,}  Valid: {len(valid_ds):,}")

    # ---- Model ----
    model = MCDropoutClassifier(
        num_classes  = cfg.NUM_CLASSES,
        dropout_rate = cfg.CLS_DROPOUT,
        pretrained   = True,
    ).to(device)

    # ---- Class-weighted BCE loss ----
    pos_weights = compute_class_weights(train_ds).to(device)
    criterion   = nn.BCEWithLogitsLoss(pos_weight=pos_weights)

    # ---- Optimiser ----
    optimizer = optim.AdamW(
        model.parameters(),
        lr           = cfg.CLS_LR,
        weight_decay = cfg.CLS_WEIGHT_DECAY,
    )
    scheduler = CosineAnnealingLR(optimizer, T_max=cfg.CLS_EPOCHS, eta_min=1e-6)

    # ---- Training loop ----
    best_auc   = 0.0
    log_rows   = []

    logger.info(f"Training classifier for {cfg.CLS_EPOCHS} epochs on {device}")
    for epoch in range(1, cfg.CLS_EPOCHS + 1):
        t0 = time.time()

        train_loss          = cls_train_one_epoch(model, train_loader, optimizer, criterion,
                                              device, cfg.CLS_GRAD_CLIP)
        val_loss, val_auc   = evaluate(model, valid_loader, criterion, device, cfg.LABEL_COLS)
        scheduler.step()

        elapsed = time.time() - t0
        logger.info(
            f"Epoch {epoch:3d}/{cfg.CLS_EPOCHS}  "
            f"train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  "
            f"val_AUC={val_auc:.4f}  lr={scheduler.get_last_lr()[0]:.2e}  "
            f"({elapsed:.1f}s)"
        )
        log_rows.append(dict(epoch=epoch, train_loss=train_loss,
                             val_loss=val_loss, val_auc=val_auc))

        # Save best
        if val_auc > best_auc:
            best_auc = val_auc
            ckpt_path = os.path.join(cfg.CHECKPOINT_DIR, "classifier_best.pt")
            torch.save({
                "epoch": epoch,
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "val_auc": best_auc,
            }, ckpt_path)
            logger.info(f"  ✓ New best AUC={best_auc:.4f} — checkpoint saved")

        # Periodic save
        if epoch % cfg.CLS_SAVE_EVERY == 0:
            ckpt_path = os.path.join(cfg.CHECKPOINT_DIR, f"classifier_epoch{epoch:03d}.pt")
            torch.save({"epoch": epoch, "model_state": model.state_dict()}, ckpt_path)

    # Save training log
    import csv
    log_path = os.path.join(cfg.LOG_DIR, "classifier_train_log.csv")
    with open(log_path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=log_rows[0].keys())
        w.writeheader()
        w.writerows(log_rows)
    logger.info(f"Training log saved to {log_path}")
    logger.info(f"Best validation AUC: {best_auc:.4f}")


if __name__ == "__main__":
    train_classifier_main()

### Train_Diffusion

In [ ]:
"""
Stage 2: Train DDPM diffusion model on CheXpert (source domain).

The diffusion model learns p_train(x).  At inference time it is used to
project test images back onto the training manifold via partial_correct().

Run:
    python train_diffusion.py
"""

import os
import logging
import time
from pathlib import Path

import torch
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
import numpy as np
from tqdm import tqdm

# ---------------------------------------------------------------------------
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger(__name__)
# ---------------------------------------------------------------------------


def diff_train_one_epoch(diffusion, loader, optimizer, device, grad_clip):
    diffusion.train()
    total_loss, n = 0.0, 0

    for imgs, _labels, _idx in tqdm(loader, desc="  diffusion train", leave=False):
        imgs = imgs.to(device)
        optimizer.zero_grad()
        loss = diffusion.train_step(imgs)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(diffusion.parameters(), grad_clip)
        optimizer.step()

        total_loss += loss.item() * imgs.size(0)
        n          += imgs.size(0)

    return total_loss / n


@torch.no_grad()
def eval_loss(diffusion, loader, device):
    diffusion.eval()
    total_loss, n = 0.0, 0
    for imgs, _labels, _idx in tqdm(loader, desc="  diffusion eval ", leave=False):
        imgs = imgs.to(device)
        loss = diffusion.train_step(imgs)
        total_loss += loss.item() * imgs.size(0)
        n          += imgs.size(0)
    return total_loss / n


# ---------------------------------------------------------------------------

def train_diffusion_main():
    cfg    = Config()
    device = torch.device(cfg.DEVICE)

    torch.manual_seed(cfg.SEED)
    np.random.seed(cfg.SEED)

    Path(cfg.CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)
    Path(cfg.LOG_DIR).mkdir(parents=True, exist_ok=True)

    # ---- Dataset (source domain only) ----
    logger.info("Loading CheXpert for diffusion training...")
    diff_tf = get_diffusion_transform(cfg.IMAGE_SIZE, cfg.MEAN, cfg.STD)

    train_ds = CheXpertDataset(
        csv_path   = cfg.CHEXPERT_TRAIN_CSV,
        image_root = cfg.CHEXPERT_IMAGE_ROOT,
        label_cols = cfg.LABEL_COLS,
        transform  = diff_tf,
    )
    valid_ds = CheXpertDataset(
        csv_path   = cfg.CHEXPERT_VALID_CSV,
        image_root = cfg.CHEXPERT_IMAGE_ROOT,
        label_cols = cfg.LABEL_COLS,
        transform  = diff_tf,
    )

    train_loader = get_dataloader(train_ds, cfg.DIFF_BATCH_SIZE, shuffle=True,  num_workers=4)
    valid_loader = get_dataloader(valid_ds, cfg.DIFF_BATCH_SIZE, shuffle=False, num_workers=4)
    logger.info(f"Diffusion train: {len(train_ds):,}  valid: {len(valid_ds):,}")

    # ---- Build model ----
    unet = UNet(
        in_channels   = 3,
        base_channels = cfg.DIFF_BASE_CHANNELS,
        channel_mults = cfg.DIFF_CHANNEL_MULTS,
        dropout       = cfg.DIFF_DROPOUT,
        attn_depths   = cfg.DIFF_ATTN_DEPTHS,
    )

    diffusion = DiffusionModel(
        unet       = unet,
        T          = cfg.DIFF_T,
        beta_start = cfg.DIFF_BETA_START,
        beta_end   = cfg.DIFF_BETA_END,
        schedule   = "linear",
    ).to(device)

    n_params = sum(p.numel() for p in diffusion.parameters() if p.requires_grad)
    logger.info(f"Diffusion model parameters: {n_params/1e6:.1f}M")

    # ---- Optimiser ----
    optimizer = optim.AdamW(diffusion.parameters(), lr=cfg.DIFF_LR)
    scheduler = CosineAnnealingLR(optimizer, T_max=cfg.DIFF_EPOCHS, eta_min=1e-6)

    # Resume from checkpoint if available
    best_ckpt = os.path.join(cfg.CHECKPOINT_DIR, "diffusion_best.pt")
    start_epoch = 1
    best_val_loss = float("inf")

    if os.path.exists(best_ckpt):
        logger.info(f"Resuming from {best_ckpt}")
        ckpt = torch.load(best_ckpt, map_location=device)
        diffusion.load_state_dict(ckpt["model_state"])
        start_epoch = ckpt.get("epoch", 0) + 1
        best_val_loss = ckpt.get("val_loss", float("inf"))

    # ---- Training loop ----
    log_rows = []
    logger.info(f"Training diffusion for {cfg.DIFF_EPOCHS} epochs on {device}")

    for epoch in range(start_epoch, cfg.DIFF_EPOCHS + 1):
        t0 = time.time()

        train_loss = diff_train_one_epoch(diffusion, train_loader, optimizer, device, cfg.DIFF_GRAD_CLIP)
        val_loss   = eval_loss(diffusion, valid_loader, device)
        scheduler.step()

        elapsed = time.time() - t0
        logger.info(
            f"Epoch {epoch:3d}/{cfg.DIFF_EPOCHS}  "
            f"train_loss={train_loss:.5f}  val_loss={val_loss:.5f}  "
            f"lr={scheduler.get_last_lr()[0]:.2e}  ({elapsed:.1f}s)"
        )
        log_rows.append(dict(epoch=epoch, train_loss=train_loss, val_loss=val_loss))

        # Save best
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save({
                "epoch": epoch,
                "model_state": diffusion.state_dict(),
                "val_loss": best_val_loss,
                "config": {
                    "T":              cfg.DIFF_T,
                    "base_channels":  cfg.DIFF_BASE_CHANNELS,
                    "channel_mults":  cfg.DIFF_CHANNEL_MULTS,
                    "attn_depths":    cfg.DIFF_ATTN_DEPTHS,
                },
            }, best_ckpt)
            logger.info(f"  ✓ Best val_loss={best_val_loss:.5f} — checkpoint saved")

        # Periodic checkpoint
        if epoch % cfg.DIFF_SAVE_EVERY == 0:
            ckpt_path = os.path.join(cfg.CHECKPOINT_DIR, f"diffusion_epoch{epoch:03d}.pt")
            torch.save({"epoch": epoch, "model_state": diffusion.state_dict()}, ckpt_path)

    # Save log
    import csv
    log_path = os.path.join(cfg.LOG_DIR, "diffusion_train_log.csv")
    with open(log_path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=log_rows[0].keys())
        w.writeheader()
        w.writerows(log_rows)
    logger.info(f"Training log saved to {log_path}")
    logger.info(f"Best validation loss: {best_val_loss:.5f}")


if __name__ == "__main__":
    train_diffusion_main()

### Factor Direction Estimator for CD-DSD

In [ ]:
"""
Learnable Factor Attribution for CD-DSD.

Trains a small MLP to classify which domain factor is responsible
for the shift between a test image and the source distribution.

Input:  feature difference  φ(x_test) - μ_CheXpert  (1024-dim)
Output: probability over 4 factors (brightness, noise, scanner, structure)

Training data is generated synthetically from CheXpert with known labels.
This makes attribution genuinely learnable — optimised by cross-entropy loss.
"""

import os
import logging
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from pathlib import Path
from torch.utils.data import DataLoader, TensorDataset

logger = logging.getLogger(__name__)

FACTOR_NAMES = [
    "brightness_contrast",
    "noise_texture",
    "scanner_artefact",
    "global_structure",
]

# ---------------------------------------------------------------------------
# Corruption functions
# ---------------------------------------------------------------------------

def corrupt_brightness(x, strength=None):
    s = strength or (torch.rand(1).item() * 1.5 + 0.5)
    return (x + s).clamp(-3, 3)

def corrupt_darkness(x, strength=None):
    s = strength or (torch.rand(1).item() * 1.5 + 0.5)
    return (x - s).clamp(-3, 3)

def corrupt_noise(x, sigma=None):
    s = sigma or (torch.rand(1).item() * 0.6 + 0.2)
    return x + torch.randn_like(x) * s

def corrupt_blur(x, kernel_size=9, sigma=None):
    s      = sigma or (torch.rand(1).item() * 2.5 + 1.0)
    k      = kernel_size
    coords = torch.arange(k, dtype=x.dtype, device=x.device) - k // 2
    g      = torch.exp(-0.5 * (coords / s) ** 2)
    g      = g / g.sum()
    kernel = (g[:, None] * g[None, :])
    kernel = kernel[None, None].expand(x.shape[1], 1, -1, -1)
    return F.conv2d(x, kernel, padding=k // 2, groups=x.shape[1])

def corrupt_contrast(x, factor=None):
    f    = factor or (torch.rand(1).item() * 2.0 + 1.5)
    mean = x.mean(dim=(-2,-1), keepdim=True)
    return ((x - mean) * f + mean).clamp(-3, 3)

def corrupt_structure(x):
    B = x.shape[0]
    theta = torch.zeros(B, 2, 3, device=x.device)
    theta[:, 0, 0] = 1.0
    theta[:, 1, 1] = 1.0
    theta[:, 0, 2] = (torch.rand(B, device=x.device) - 0.5) * 0.4
    theta[:, 1, 2] = (torch.rand(B, device=x.device) - 0.5) * 0.4
    grid = F.affine_grid(theta, x.shape, align_corners=False)
    return F.grid_sample(x, grid, align_corners=False)

# Map factor index to list of corruption functions
# Multiple corruptions per factor increases training diversity
FACTOR_CORRUPTIONS = {
    0: [corrupt_brightness, corrupt_darkness, corrupt_contrast],   # brightness_contrast
    1: [corrupt_noise],                           # noise_texture
    2: [corrupt_blur],                            # scanner_artefact
    3: [corrupt_structure],                       # global_structure
}


# ---------------------------------------------------------------------------
# Learnable Factor MLP
# ---------------------------------------------------------------------------

class FactorMLP(nn.Module):
    """
    Small MLP trained to identify dominant domain shift factor.

    Input:  normalised feature difference (1024-dim)
    Output: logits over 4 factors
    """
    def __init__(self, feat_dim=1024, hidden_dim=256, n_factors=4,
                 dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(feat_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, n_factors),
        )

    def forward(self, x):
        return self.net(x)   # (B, n_factors) logits


# ---------------------------------------------------------------------------
# Dataset builder
# ---------------------------------------------------------------------------

@torch.no_grad()
def build_training_data(diagnoser, cfg, n_samples_per_factor=500):
    """
    Generate synthetic training data:
      - For each factor, apply its corruption to CheXpert images
      - Extract feature difference φ(x_corrupted) - μ_CheXpert
      - Label = factor index

    Returns
    -------
    X : (N, 1024) normalised feature differences
    y : (N,)      factor labels  0-3
    """

    logger.info("Building factor classifier training data...")
    eval_tf = get_eval_transform(
        cfg.IMAGE_SIZE, cfg.MEAN, cfg.STD)
    ds = CheXpertDataset(
        csv_path   = cfg.CHEXPERT_TRAIN_CSV,
        image_root = cfg.CHEXPERT_IMAGE_ROOT,
        label_cols = cfg.LABEL_COLS,
        transform  = eval_tf,
    )
    loader = DataLoader(ds, batch_size=16, shuffle=True,
                        collate_fn=collate_fn, num_workers=2)

    # Collect enough clean images
    n_total = n_samples_per_factor * 4
    clean_batches = []
    for imgs, _, _ in loader:
        clean_batches.append(imgs.to(diagnoser.device))
        if sum(len(b) for b in clean_batches) >= n_total:
            break
    clean_imgs = torch.cat(clean_batches)[:n_total]
    logger.info(f"  Collected {len(clean_imgs)} clean images")

    all_X = []
    all_y = []

    for factor_idx, corrupt_fns in FACTOR_CORRUPTIONS.items():
        factor_name = FACTOR_NAMES[factor_idx]
        logger.info(f"  Generating samples for: {factor_name}")

        # Take a slice of clean images for this factor
        start = factor_idx * n_samples_per_factor
        end   = start + n_samples_per_factor
        imgs  = clean_imgs[start:end]

        # Apply corruptions (round-robin if multiple)
        corrupted_list = []
        for i, img in enumerate(imgs):
            fn = corrupt_fns[i % len(corrupt_fns)]
            corrupted_list.append(fn(img.unsqueeze(0)).squeeze(0))
        corrupted = torch.stack(corrupted_list)   # (N, 3, H, W)

        # Extract features in batches
        feat_diffs = []
        batch_size = 16
        for i in range(0, len(corrupted), batch_size):
            batch = corrupted[i:i + batch_size]
            feats = diagnoser._extract_features(batch)          # (B, 1024)
            # Normalised feature difference from source distribution
            diff  = ((feats - diagnoser.chexpert_mean)
                     / diagnoser.chexpert_std)                  # (B, 1024)
            feat_diffs.append(diff.cpu())

        feat_diffs = torch.cat(feat_diffs)                      # (N, 1024)
        labels     = torch.full((len(feat_diffs),), factor_idx,
                                dtype=torch.long)

        all_X.append(feat_diffs)
        all_y.append(labels)

        logger.info(f"    {factor_name}: {len(feat_diffs)} samples")

    X = torch.cat(all_X)   # (4N, 1024)
    y = torch.cat(all_y)   # (4N,)
    logger.info(f"Training data built: {len(X)} total samples, "
                f"{len(FACTOR_NAMES)} classes")
    return X, y


# ---------------------------------------------------------------------------
# Trainer
# ---------------------------------------------------------------------------

class FactorClassifierTrainer:
    """
    Trains the FactorMLP on synthetic corrupted CheXpert data.
    """

    def __init__(self, diagnoser, cfg):
        self.diagnoser = diagnoser
        self.cfg       = cfg
        self.device    = diagnoser.device
        self.model     = FactorMLP(
            feat_dim   = 1024,
            hidden_dim = 256,
            n_factors  = len(FACTOR_NAMES),
        ).to(self.device)

    def train(self,
              n_samples_per_factor = 500,
              epochs               = 30,
              lr                   = 1e-3,
              batch_size           = 64,
              val_split            = 0.15):
        """
        Full training loop.

        Returns trained FactorMLP.
        """
        # Build dataset
        X, y = build_training_data(
            self.diagnoser, self.cfg,
            n_samples_per_factor=n_samples_per_factor)

        # Train / val split
        n_val   = int(len(X) * val_split)
        idx     = torch.randperm(len(X))
        X_train, y_train = X[idx[n_val:]], y[idx[n_val:]]
        X_val,   y_val   = X[idx[:n_val]], y[idx[:n_val]]

        train_ds = TensorDataset(X_train, y_train)
        val_ds   = TensorDataset(X_val,   y_val)
        train_loader = DataLoader(train_ds, batch_size=batch_size,
                                  shuffle=True)
        val_loader   = DataLoader(val_ds,   batch_size=batch_size)

        optimizer = torch.optim.Adam(self.model.parameters(), lr=lr,
                                     weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=epochs)

        best_val_acc = 0.0
        best_state   = None

        logger.info(f"Training FactorMLP for {epochs} epochs...")
        logger.info(f"  Train: {len(train_ds)}  Val: {len(val_ds)}")

        for epoch in range(1, epochs + 1):
            # ---- Train ----
            self.model.train()
            train_loss = 0.0
            for xb, yb in train_loader:
                xb, yb = xb.to(self.device), yb.to(self.device)
                optimizer.zero_grad()
                logits = self.model(xb)
                loss   = F.cross_entropy(logits, yb)
                loss.backward()
                optimizer.step()
                train_loss += loss.item() * len(xb)
            train_loss /= len(train_ds)

            # ---- Validate ----
            self.model.eval()
            correct = total = 0
            with torch.no_grad():
                for xb, yb in val_loader:
                    xb, yb  = xb.to(self.device), yb.to(self.device)
                    preds   = self.model(xb).argmax(dim=1)
                    correct += (preds == yb).sum().item()
                    total   += len(yb)
            val_acc = correct / total

            scheduler.step()

            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_state   = {k: v.clone()
                                for k, v in self.model.state_dict().items()}

            if epoch % 5 == 0:
                logger.info(f"  Epoch {epoch:3d}/{epochs}  "
                            f"loss={train_loss:.4f}  "
                            f"val_acc={val_acc:.1%}")

        # Restore best
        self.model.load_state_dict(best_state)
        logger.info(f"Training complete. Best val accuracy: "
                    f"{best_val_acc:.1%}")
        return self.model, best_val_acc

    def save(self, path):
        Path(path).parent.mkdir(parents=True, exist_ok=True)
        torch.save({
            "model_state": self.model.state_dict(),
            "factor_names": FACTOR_NAMES,
        }, path)
        logger.info(f"Factor classifier saved → {path}")


# ---------------------------------------------------------------------------
# Inference wrapper
# ---------------------------------------------------------------------------

class FactorAttributor:
    """
    Uses trained FactorMLP to attribute domain uncertainty to factors.
    Drop-in replacement for the heuristic direction-based attribution.
    """

    def __init__(self, diagnoser, cfg):
        self.diagnoser = diagnoser
        self.cfg       = cfg
        self.device    = diagnoser.device
        self.model     = FactorMLP(
            feat_dim   = 1024,
            hidden_dim = 256,
            n_factors  = len(FACTOR_NAMES),
        ).to(self.device)
        self.model.eval()

    def load(self, path):
        ckpt = torch.load(path, map_location=self.device)
        self.model.load_state_dict(ckpt["model_state"])
        self.model.eval()
        logger.info(f"Factor classifier loaded from {path}")
        return self

    @torch.no_grad()
    def attribute(self, x_test):
        """
        Returns per-factor attribution probabilities.

        Returns
        -------
        dict  factor_name → (B,) probability tensor
        """
        feats     = self.diagnoser._extract_features(x_test)
        feat_diff = ((feats - self.diagnoser.chexpert_mean)
                     / self.diagnoser.chexpert_std)

        logits = self.model(feat_diff)              # (B, 4)
        probs  = F.softmax(logits, dim=1)           # (B, 4)

        return {
            FACTOR_NAMES[i]: probs[:, i]
            for i in range(len(FACTOR_NAMES))
        }

    @torch.no_grad()
    def attribute_with_uncertainty(self, x_test, n_samples=10):
        """
        MC Dropout on the factor classifier for attribution uncertainty.
        Returns mean ± std per factor per image.
        """
        # Enable dropout for uncertainty
        for m in self.model.modules():
            if isinstance(m, nn.Dropout):
                m.train()

        feats     = self.diagnoser._extract_features(x_test)
        feat_diff = ((feats - self.diagnoser.chexpert_mean)
                     / self.diagnoser.chexpert_std)

        all_probs = []
        for _ in range(n_samples):
            logits = self.model(feat_diff)
            all_probs.append(F.softmax(logits, dim=1))

        self.model.eval()

        probs_stack = torch.stack(all_probs)        # (n_samples, B, 4)
        mean_probs  = probs_stack.mean(dim=0)       # (B, 4)
        std_probs   = probs_stack.std(dim=0)        # (B, 4)

        mean_attrs = {FACTOR_NAMES[i]: mean_probs[:, i]
                      for i in range(len(FACTOR_NAMES))}
        std_attrs  = {FACTOR_NAMES[i]: std_probs[:, i]
                      for i in range(len(FACTOR_NAMES))}

        return mean_attrs, std_attrs

### CD_DSD

In [ ]:
import os
import logging
from pathlib import Path
from typing import Optional

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from torch.utils.data import DataLoader

logger = logging.getLogger(__name__)


# ---------------------------------------------------------------------------
# Tensor ↔ display image helpers
# ---------------------------------------------------------------------------

_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)


def denorm(t: torch.Tensor) -> np.ndarray:
    img = (t.cpu().float() * _STD + _MEAN).clamp(0, 1)
    return (img.permute(1, 2, 0).numpy() * 255).astype(np.uint8)


# ---------------------------------------------------------------------------
# Main Diagnoser class
# ---------------------------------------------------------------------------

class CDDSDDiagnoser:
    """
    U_domain now comes directly from DiffusionModel.domain_score(): the
    diffusion model's own denoising error on the test image. The model was
    trained only on CheXpert, so its noise predictions are accurate for
    in-domain images and degrade for out-of-domain images — that prediction
    gap IS the domain-shift signal, no SDEdit round-trip or feature-distance
    proxy required. This keeps the diffusion model load-bearing for the
    actual measurement, not just for the visualization panel.
    """

    def __init__(self, cfg):
        self.cfg    = cfg
        self.device = torch.device(cfg.DEVICE)
        self.classifier = self._load_classifier()
        self.diffusion  = self._load_diffusion()
        self._calibrate()
        self.factor_attributor = self._load_factor_attributor()

    def _load_classifier(self):
        ckpt_path = os.path.join(self.cfg.CHECKPOINT_DIR, self.cfg.CLS_CKPT_NAME)
        model = MCDropoutClassifier(
            num_classes  = self.cfg.NUM_CLASSES,
            dropout_rate = self.cfg.CLS_DROPOUT,
            pretrained   = False,
        ).to(self.device)
        if os.path.exists(ckpt_path):
            ckpt = torch.load(ckpt_path, map_location=self.device)
            model.load_state_dict(ckpt["model_state"])
            logger.info(f"Classifier loaded from {ckpt_path}  "
                        f"(best AUC={ckpt.get('val_auc', '?')})")
        else:
            raise FileNotFoundError(
                f"Classifier checkpoint NOT found at {ckpt_path}. "
                f"Refusing to silently fall back to random weights — "
                f"check cfg.CLS_CKPT_NAME / cfg.CHECKPOINT_DIR."
            )
        return model

    def _load_diffusion(self):
        ckpt_path = os.path.join(self.cfg.CHECKPOINT_DIR, "diffusion_best.pt")
        unet = UNet(
            in_channels   = 3,
            base_channels = self.cfg.DIFF_BASE_CHANNELS,
            channel_mults = self.cfg.DIFF_CHANNEL_MULTS,
            dropout       = self.cfg.DIFF_DROPOUT,
            attn_depths   = self.cfg.DIFF_ATTN_DEPTHS,
        )
        model = DiffusionModel(
            unet       = unet,
            T          = self.cfg.DIFF_T,
            beta_start = self.cfg.DIFF_BETA_START,
            beta_end   = self.cfg.DIFF_BETA_END,
        ).to(self.device)
        if os.path.exists(ckpt_path):
            ckpt = torch.load(ckpt_path, map_location=self.device)
            model.load_state_dict(ckpt["model_state"])
            logger.info(f"Diffusion model loaded from {ckpt_path}")
        else:
            raise FileNotFoundError(f"Diffusion checkpoint NOT found at {ckpt_path}.")
        return model

    @torch.no_grad()
    def _calibrate(self, n_samples=200):
        """
        Calibration pass on clean, known-in-domain CheXpert images.
        Computes:
          - chexpert_mean / chexpert_std   : classifier feature stats,
                                              used only as FactorMLP input.
          - domain_baseline_mean / _std    : distribution of the diffusion
                                              denoising-error score on
                                              in-domain images.
          - domain_weight                  : calibrated (not hand-tuned)
                                              scale factor so the z-scored
                                              domain signal has comparable
                                              spread to semantic uncertainty
                                              on in-domain data.
        """
        logger.info("Calibrating on clean CheXpert reference set...")
        eval_tf = get_eval_transform(self.cfg.IMAGE_SIZE, self.cfg.MEAN, self.cfg.STD)
        ds = CheXpertDataset(
            csv_path   = self.cfg.CHEXPERT_TRAIN_CSV,
            image_root = self.cfg.CHEXPERT_IMAGE_ROOT,
            label_cols = self.cfg.LABEL_COLS,
            transform  = eval_tf,
        )
        loader = DataLoader(ds, batch_size=16, shuffle=True,
                            collate_fn=collate_fn, num_workers=2)

        all_feats, all_domain_raw, all_semantic = [], [], []
        n_collected = 0
        for imgs, _, _ in loader:
            imgs = imgs.to(self.device)
            all_feats.append(self._extract_features(imgs))
            all_domain_raw.append(self._domain_score_raw(imgs))
            all_semantic.append(self._uncertainty(imgs))
            n_collected += imgs.shape[0]
            if n_collected >= n_samples:
                break

        feats = torch.cat(all_feats)[:n_samples]
        self.chexpert_mean = feats.mean(0)
        self.chexpert_std  = feats.std(0).clamp(min=1e-6)

        domain_raw = torch.cat(all_domain_raw)[:n_samples]
        self.domain_baseline_mean = domain_raw.mean()
        self.domain_baseline_std  = domain_raw.std().clamp(min=1e-6)

        semantic = torch.cat(all_semantic)[:n_samples]
        # Calibrated weight: match domain-signal spread to semantic-signal
        # spread on in-domain data, instead of a hand-picked constant.
        self.domain_weight = semantic.std().clamp(min=1e-6)

        logger.info(
            f"Calibration done. domain_baseline_mean={self.domain_baseline_mean.item():.5f}  "
            f"domain_baseline_std={self.domain_baseline_std.item():.5f}  "
            f"semantic_std={semantic.std().item():.5f}  "
            f"domain_weight={self.domain_weight.item():.5f}"
        )

    def _load_factor_attributor(self):
        ckpt_path  = os.path.join(self.cfg.CHECKPOINT_DIR, "factor_classifier.pt")
        attributor = FactorAttributor(self, self.cfg)
        if os.path.exists(ckpt_path):
            attributor.load(ckpt_path)
        else:
            logger.info("Factor classifier not found — training now...")
            trainer = FactorClassifierTrainer(self, self.cfg)
            trainer.train(n_samples_per_factor=500, epochs=30)
            trainer.save(ckpt_path)
            attributor.load(ckpt_path)
        return attributor

    @torch.no_grad()
    def _extract_features(self, x):
        self.classifier.eval()
        feats = self.classifier.features(x)
        feats = torch.nn.functional.relu(feats, inplace=False)
        feats = self.classifier.pool(feats).flatten(1)  # (B, 1024)
        return feats

    @torch.no_grad()
    def _domain_score_raw(self, x):
        """
        Diffusion denoising-error domain score, averaged over repeated
        noise draws for stability (each draw uses fresh random noise at
        each sampled timestep, so single-draw estimates are noisy).
        """
        scores = [
            self.diffusion.domain_score(x, n_timesteps=self.cfg.DOMAIN_SCORE_TIMESTEPS)
            for _ in range(self.cfg.DOMAIN_SCORE_REPEATS)
        ]
        return torch.stack(scores).mean(dim=0)   # (B,)

    def _uncertainty(self, x: torch.Tensor) -> torch.Tensor:
        """MC Dropout predictive entropy on the ORIGINAL image."""
        return self.classifier.uncertainty_scalar(x, n_samples=self.cfg.MC_SAMPLES)

    @torch.no_grad()
    def diagnose_batch(self, x_test, domain="unknown", save_vis=True,
                       save_dir=None, sample_ids=None):
        x_test   = x_test.to(self.device)
        B        = x_test.shape[0]
        save_dir = save_dir or self.cfg.RESULTS_DIR
        Path(save_dir).mkdir(parents=True, exist_ok=True)
        if sample_ids is None:
            sample_ids = list(range(B))

        # ----------------------------------------------------------------
        # Step 1: Semantic uncertainty — MC Dropout on the original image
        # ----------------------------------------------------------------
        u_semantic = self._uncertainty(x_test)          # (B,)

        # ----------------------------------------------------------------
        # Step 2: Domain uncertainty — diffusion denoising error,
        #   z-scored against the clean-CheXpert calibration baseline,
        #   then scaled by the calibrated (not hand-tuned) domain_weight.
        # ----------------------------------------------------------------
        domain_raw    = self._domain_score_raw(x_test)                       # (B,)
        domain_z      = ((domain_raw - self.domain_baseline_mean)
                          / self.domain_baseline_std).clamp(min=0.0)
        u_domain      = self.domain_weight * domain_z                        # (B,) calibrated units

        u_total         = u_semantic + u_domain
        domain_fraction = u_domain / (u_total + 1e-7)

        # ----------------------------------------------------------------
        # Step 3: Full correction image — for visualization ONLY.
        #   Does not feed any of the scores above.
        # ----------------------------------------------------------------
        x_star = self.diffusion.partial_correct(
            x_test, self.cfg.T_STAR_FULL,
            num_steps=self.cfg.DDIM_STEPS, eta=self.cfg.DDIM_ETA
        )

        # ----------------------------------------------------------------
        # Step 4: Factor attribution — learned FactorMLP classifier.
        #   Independent diagnostic layer; not part of the U_domain score.
        # ----------------------------------------------------------------
        level_names = sorted(self.cfg.T_STAR_LEVELS,
                             key=lambda k: self.cfg.T_STAR_LEVELS[k])
        level_data  = {}
        for factor_name in level_names:
            t_k = self.cfg.T_STAR_LEVELS[factor_name]
            if factor_name == "brightness_contrast":
                x_k = self.diffusion.brightness_correct(x_test)
            else:
                x_k = self.diffusion.partial_correct(
                    x_test, t_k,
                    num_steps = self.cfg.DDIM_STEPS,
                    eta       = self.cfg.DDIM_ETA,
                )
            level_data[factor_name] = dict(x=x_k)

        mean_attrs, std_attrs = self.factor_attributor.attribute_with_uncertainty(
            x_test, n_samples=10)

        factor_attributions_batch = [{} for _ in range(B)]
        for factor_name in level_names:
            attr_k = mean_attrs[factor_name]   # (B,)
            for b in range(B):
                factor_attributions_batch[b][factor_name] = attr_k[b].item()

        # ----------------------------------------------------------------
        # Step 5: Predictions before/after the visualization correction
        # ----------------------------------------------------------------
        pred_orig = self.classifier.predict(x_test)
        pred_corr = self.classifier.predict(x_star)

        # ----------------------------------------------------------------
        # Compile results
        # ----------------------------------------------------------------
        results = []
        for b in range(B):
            result = dict(
                sample_id           = sample_ids[b],
                domain              = domain,
                u_total             = u_total[b].item(),
                u_domain            = u_domain[b].item(),
                u_domain_raw        = domain_raw[b].item(),
                u_semantic          = u_semantic[b].item(),
                domain_fraction     = domain_fraction[b].item(),
                factor_attributions = factor_attributions_batch[b],
                pred_original       = pred_orig[b].cpu().tolist(),
                pred_corrected      = pred_corr[b].cpu().tolist(),
                label_cols          = self.cfg.LABEL_COLS,
            )
            if save_vis:
                vis_path = os.path.join(
                    save_dir, f"{domain}_sample{sample_ids[b]}_diagnosis.png")
                self._save_visualisation(
                    x_orig       = x_test[b],
                    x_corrected  = x_star[b],
                    level_images = {k: v["x"][b] for k, v in level_data.items()},
                    result       = result,
                    save_path    = vis_path,
                )
                result["vis_path"] = vis_path
            results.append(result)

        return results

    def _save_visualisation(self, x_orig, x_corrected, level_images,
                            result, save_path):
        n_levels   = len(level_images)
        total_imgs = 2 + n_levels
        fig_w      = 3 * total_imgs + 2
        fig        = plt.figure(figsize=(fig_w, 7))
        gs         = gridspec.GridSpec(2, total_imgs, figure=fig,
                                       hspace=0.4, wspace=0.3)

        # Row 0: images
        ax = fig.add_subplot(gs[0, 0])
        ax.imshow(denorm(x_orig), cmap="gray")
        ax.set_title("Original\n(test)", fontsize=9)
        ax.axis("off")

        for i, (name, x_k) in enumerate(level_images.items()):
            ax = fig.add_subplot(gs[0, i + 1])
            ax.imshow(denorm(x_k), cmap="gray")
            ax.set_title(f"{name.replace('_', chr(10))}\ncorrection (illustrative)", fontsize=8)
            ax.axis("off")

        ax = fig.add_subplot(gs[0, -1])
        ax.imshow(denorm(x_corrected), cmap="gray")
        ax.set_title("Full\ncorrection\n(x*, illustrative)", fontsize=9)
        ax.axis("off")

        # Row 1 left: uncertainty decomposition bar
        ax_unc = fig.add_subplot(gs[1, :total_imgs // 2])
        ax_unc.bar(
            ["Domain\nUncertainty\n(diffusion denoising error)",
             "Semantic\nUncertainty\n(MC Dropout)"],
            [result["u_domain"], result["u_semantic"]],
            color=["#E74C3C", "#3498DB"], edgecolor="k", width=0.5
        )
        ax_unc.set_title(
            f"Total U={result['u_total']:.3f}  "
            f"Domain fraction={result['domain_fraction']:.1%}",
            fontsize=9)
        ax_unc.set_ylabel("Uncertainty")

        # Row 1 right: factor attribution (learned FactorMLP probabilities)
        ax_attr = fig.add_subplot(gs[1, total_imgs // 2:])
        attrs = result["factor_attributions"]
        ax_attr.barh(
            [k.replace("_", "\n") for k in attrs],
            list(attrs.values()),
            color="#2ECC71", edgecolor="k"
        )
        ax_attr.set_title("Factor Attribution\n(learned FactorMLP probability)", fontsize=9)
        ax_attr.set_xlabel("P(factor)")

        fig.suptitle(
            f"CD-DSD  |  domain={result['domain']}  |  sample={result['sample_id']}",
            fontsize=10, fontweight="bold")
        plt.savefig(save_path, dpi=120, bbox_inches="tight")
        plt.close(fig)


# ---------------------------------------------------------------------------
# Evaluate a full domain dataset
# ---------------------------------------------------------------------------

def evaluate_domain(diagnoser, dataset, domain_name, max_samples, batch_size=8):
    loader = DataLoader(
        dataset,
        batch_size  = batch_size,
        shuffle     = False,
        collate_fn  = collate_fn,
        num_workers = 4,
        pin_memory  = True,
    )

    all_results = []
    n_done      = 0

    for imgs, _labels, indices in loader:
        if n_done >= max_samples:
            break
        cap     = min(imgs.shape[0], max_samples - n_done)
        imgs    = imgs[:cap]
        ids     = list(indices[:cap])
        results = diagnoser.diagnose_batch(
            imgs, domain=domain_name, save_vis=True, sample_ids=ids)
        all_results.extend(results)
        n_done += cap
        logger.info(f"  {domain_name}: {n_done}/{max_samples} diagnosed")

    return all_results

### Main

In [ ]:
import csv
import json
import logging
import os
from pathlib import Path

import numpy as np
import torch

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger(__name__)


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def setup_dirs(cfg):
    for d in [cfg.CHECKPOINT_DIR, cfg.LOG_DIR, cfg.PLOT_DIR, cfg.RESULTS_DIR]:
        Path(d).mkdir(parents=True, exist_ok=True)
    logger.info(f"Output root: {cfg.OUTPUT_DIR}")


def save_results_csv(results, path):
    if not results:
        return
    rows = []
    for r in results:
        row = {k: v for k, v in r.items()
               if k not in ("factor_attributions", "pred_original",
                            "pred_corrected", "label_cols", "vis_path")}
        row.update({f"attr_{k}": v
                    for k, v in r.get("factor_attributions", {}).items()})
        label_cols = r.get("label_cols", [])
        for c, label in enumerate(label_cols):
            row[f"pred_orig_{label}"] = round(r["pred_original"][c], 4)
            row[f"pred_corr_{label}"] = round(r["pred_corrected"][c], 4)
        rows.append(row)

    with open(path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=rows[0].keys())
        w.writeheader()
        w.writerows(rows)
    logger.info(f"Results saved → {path}")


def print_summary(results, domain):
    if not results:
        return
    u_total    = np.mean([r["u_total"]         for r in results])
    u_domain   = np.mean([r["u_domain"]        for r in results])
    u_semantic = np.mean([r["u_semantic"]      for r in results])
    dom_frac   = np.mean([r["domain_fraction"] for r in results])

    logger.info(f"\n{'='*60}")
    logger.info(f"  CD-DSD Summary — Domain: {domain}")
    logger.info(f"  Samples diagnosed : {len(results)}")
    logger.info(f"  Mean U_total      : {u_total:.4f}")
    logger.info(f"  Mean U_domain     : {u_domain:.4f}  ({dom_frac:.1%} of total)")
    logger.info(f"  Mean U_semantic   : {u_semantic:.4f}")

    if results[0].get("factor_attributions"):
        logger.info("  Factor Attribution (mean):")
        for f in results[0]["factor_attributions"].keys():
            mean_attr = np.mean([r["factor_attributions"][f] for r in results])
            logger.info(f"    {f:<30s}: {mean_attr:.4f}")
    logger.info('='*60)


# ---------------------------------------------------------------------------
# Stage runners  (no imports — classes already defined above in notebook)
# ---------------------------------------------------------------------------

# def run_train_classifier():
#     logger.info("\n" + "="*60)
#     logger.info("  STAGE 1: Training Classifier")
#     logger.info("="*60)
#     train_classifier_main()


# def run_train_diffusion():
#     logger.info("\n" + "="*60)
#     logger.info("  STAGE 2: Training Diffusion Model")
#     logger.info("="*60)
#     train_diffusion_main()


def run_diagnose(cfg):
    eval_tf   = get_eval_transform(cfg.IMAGE_SIZE, cfg.MEAN, cfg.STD)
    diagnoser = CDDSDDiagnoser(cfg)
    domains   = []

    if os.path.exists(cfg.CHEXPERT_VALID_CSV):
        chex_ds = CheXpertDataset(
            csv_path   = cfg.CHEXPERT_VALID_CSV,
            image_root = cfg.CHEXPERT_IMAGE_ROOT,
            label_cols = cfg.LABEL_COLS,
            transform  = eval_tf,
        )
        domains.append(("CheXpert-valid", chex_ds))
    
    if os.path.exists(cfg.MIMIC_VALID_CSV):
        mimic_ds = MIMICCXRDataset(
            csv_path   = cfg.MIMIC_VALID_CSV,
            image_root = cfg.MIMIC_IMAGE_ROOT,
            label_cols = cfg.LABEL_COLS,
            transform  = eval_tf,
            max_samples= cfg.MAX_DIAG_SAMPLES * 2,
        )
        domains.append(("MIMIC-CXR", mimic_ds))
    else:
        logger.warning(f"MIMIC valid CSV not found: {cfg.MIMIC_VALID_CSV}")

    if os.path.exists(cfg.NIH_CSV):
        nih_ds = NIHChestXrayDataset(
            csv_path   = cfg.NIH_CSV,
            image_root = cfg.NIH_IMAGE_ROOT,
            label_cols = cfg.LABEL_COLS,
            transform  = eval_tf,
            max_samples= cfg.MAX_DIAG_SAMPLES * 2,
        )
        domains.append(("NIH-ChestXray14", nih_ds))
    else:
        logger.warning(f"NIH CSV not found: {cfg.NIH_CSV}")

    # if os.path.exists(cfg.CHEXPERT_VALID_CSV):
    #     chex_ds = CheXpertDataset(
    #         csv_path   = cfg.CHEXPERT_VALID_CSV,
    #         image_root = cfg.CHEXPERT_IMAGE_ROOT,
    #         label_cols = cfg.LABEL_COLS,
    #         transform  = eval_tf,
    #     )
    #     domains.append(("CheXpert-valid", chex_ds))

    for domain_name, dataset in domains:
        logger.info(f"\nRunning CD-DSD on {domain_name}  "
                    f"(n={min(cfg.MAX_DIAG_SAMPLES, len(dataset))})")
        results = evaluate_domain(
            diagnoser   = diagnoser,
            dataset     = dataset,
            domain_name = domain_name,
            max_samples = cfg.MAX_DIAG_SAMPLES,
            batch_size  = 8,
        )

        csv_path = os.path.join(cfg.RESULTS_DIR, f"cd_dsd_{domain_name}.csv")
        save_results_csv(results, csv_path)

        json_path = os.path.join(cfg.RESULTS_DIR, f"cd_dsd_{domain_name}_summary.json")
        with open(json_path, "w") as f:
            summary = dict(
                domain          = domain_name,
                n_samples       = len(results),
                mean_u_total    = float(np.mean([r["u_total"]         for r in results])),
                mean_u_domain   = float(np.mean([r["u_domain"]        for r in results])),
                mean_u_semantic = float(np.mean([r["u_semantic"]      for r in results])),
                mean_dom_frac   = float(np.mean([r["domain_fraction"] for r in results])),
            )
            json.dump(summary, f, indent=2)

        print_summary(results, domain_name)


# ---------------------------------------------------------------------------
# Run
# ---------------------------------------------------------------------------

cfg = Config()
setup_dirs(cfg)
torch.manual_seed(cfg.SEED)
np.random.seed(cfg.SEED)

# run_train_classifier()
# run_train_diffusion()
run_diagnose(cfg)

---
## ⚡ Quick Setup — Start Here (skip cells 0–18)

Run **only this cell** and the experiment cells below.  
Cells 0–18 define the same classes now packaged in `cd_dsd/`; no need to re-run them.

In [4]:
import sys, os, logging
sys.path.insert(0, "/home/dawood/lab2_rotaion/counterfactual_diff_uncertainty")

from cd_dsd import (
    Config,
    CheXpertDataset, MIMICCXRDataset, NIHChestXrayDataset,
    get_eval_transform, collate_fn,
    CDDSDDiagnoser, evaluate_domain, denorm, hf_energy_score,
    corrupt_brightness, corrupt_darkness, corrupt_noise, corrupt_blur,
    corrupt_contrast, corrupt_structure, FACTOR_NAMES,
    BaselineSuite, setup_dirs,
)
from sklearn.metrics import roc_auc_score
import numpy as np, torch

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("notebook")

cfg = Config()
setup_dirs(cfg)

logger.info("Loading CDDSDDiagnoser (classifier + diffusion + HFER calibration)...")
diagnoser = CDDSDDiagnoser(cfg)

logger.info("Fitting BaselineSuite (ReAct/DICE/ViM/KNN on 1000 CheXpert images)...")
suite = BaselineSuite(diagnoser, cfg)
suite.fit(n_samples=1000)

logger.info("Quick Setup complete — skip cells 0-18, run experiment cells below.")

12:22:36 INFO cd_dsd.utils: Output root: /home/dawood/lab2_rotaion/counterfactual_diff_uncertainty/
12:22:36 INFO notebook: Loading CDDSDDiagnoser (classifier + diffusion + HFER calibration)...
12:22:44 INFO cd_dsd.diagnoser: Classifier loaded from /home/dawood/lab2_rotaion/counterfactual_diff_uncertainty/checkpoints/classifier_best.pt  (best AUC=0.9441657501893854)
12:22:45 INFO cd_dsd.diagnoser: Diffusion model loaded from /home/dawood/lab2_rotaion/counterfactual_diff_uncertainty/checkpoints/diffusion_best.pt
12:22:45 INFO cd_dsd.diagnoser: Calibrating on clean CheXpert reference set...
12:22:45 INFO cd_dsd.datasets: Validating CheXpertDataset...
12:22:52 INFO cd_dsd.datasets:   Valid: 47561/47561 images (100.0%)
12:24:13 INFO cd_dsd.diagnoser: Calibration done. domain_baseline_mean=0.01035  domain_baseline_std=0.00238  hfer_mean=0.02116  hfer_std=0.00611  domain_weight=0.09211
12:24:13 INFO cd_dsd.factor_mlp: Factor classifier loaded from /home/dawood/lab2_rotaion/counterfactual_dif

### Synthetic_Validation_Experiment for CD-DSD

In [2]:
"""
Synthetic Validation for CD-DSD.

1. Take clean CheXpert images (in-domain)
2. Apply one known corruption at a time (brightness, noise, blur, etc.)
3. Run CD-DSD -> check if top factor matches the applied corruption
4. Report: Attribution Accuracy + AUROC per corruption type

corrupt_* functions are imported from cd_dsd (Quick Setup).
"""
import os, logging, numpy as np, torch
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader

plt.rcParams.update({"axes.titlesize":15,"axes.labelsize":13,"xtick.labelsize":11,
                     "ytick.labelsize":11,"axes.titlecolor":"black",
                     "axes.labelcolor":"black","xtick.color":"black","ytick.color":"black"})

logger = logging.getLogger("synthetic_val")

# Corruption registry — maps name -> (function, expected top factor)
CORRUPTIONS = {
    "brightness": (corrupt_brightness, "brightness_contrast"),
    "darkness":   (corrupt_darkness,   "brightness_contrast"),
    "noise":      (corrupt_noise,       "noise_texture"),
    "blur":       (corrupt_blur,        "scanner_artefact"),
    "contrast":   (corrupt_contrast,    "brightness_contrast"),
    "combined":   (lambda x: corrupt_blur(corrupt_noise(corrupt_brightness(x, 0.3), 0.2), 7, 1.5),
                   "scanner_artefact"),
}

SV_N     = 80
SV_BATCH = 16
SV_DIR   = os.path.join(cfg.RESULTS_DIR, "synthetic_validation")
Path(SV_DIR).mkdir(parents=True, exist_ok=True)

eval_tf  = get_eval_transform(cfg.IMAGE_SIZE, cfg.MEAN, cfg.STD)
chex_ds  = CheXpertDataset(cfg.CHEXPERT_VALID_CSV, cfg.CHEXPERT_IMAGE_ROOT,
                            cfg.LABEL_COLS, eval_tf)
loader   = DataLoader(chex_ds, batch_size=SV_BATCH, shuffle=True,
                      collate_fn=collate_fn, num_workers=2)
device   = diagnoser.device

clean_batches = []
for imgs, _, _ in loader:
    clean_batches.append(imgs)
    if sum(len(b) for b in clean_batches) >= SV_N: break
clean_images = torch.cat(clean_batches)[:SV_N].to(device)

attr_acc = {}
auroc_sv = {}

for c_name, (c_fn, expected_factor) in CORRUPTIONS.items():
    x_c = c_fn(clean_images.clone())
    n_correct = 0

    clean_scores, corr_scores = [], []
    with torch.no_grad():
        for i in range(0, len(x_c), SV_BATCH):
            batch_c = x_c[i:i+SV_BATCH]
            batch_cl= clean_images[i:i+SV_BATCH]
            # Attribution accuracy
            results = diagnoser.diagnose_batch(batch_c, domain=c_name,
                                               save_vis=False, save_dir=SV_DIR)
            for r in results:
                top = max(r["factor_attributions"], key=r["factor_attributions"].get)
                if top == expected_factor: n_correct += 1
            # AUROC scores
            corr_scores.append(suite.score_cd_dsd(batch_c).cpu().numpy())
            clean_scores.append(suite.score_cd_dsd(batch_cl).cpu().numpy())

    attr_acc[c_name] = n_correct / len(x_c)
    clean_s = np.concatenate(clean_scores)
    corr_s  = np.concatenate(corr_scores)
    lbl     = np.array([0]*len(clean_s) + [1]*len(corr_s))
    try:    auroc_sv[c_name] = roc_auc_score(lbl, np.concatenate([clean_s, corr_s]))
    except: auroc_sv[c_name] = float("nan")
    logger.info(f"  {c_name:<12} attr_acc={attr_acc[c_name]:.2f}  AUROC={auroc_sv[c_name]:.3f}")

# -- Figure (double column, 13") --
c_names = list(CORRUPTIONS.keys())
x_pos   = np.arange(len(c_names))
width   = 0.35

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
bars = ax.bar(x_pos, [attr_acc[c] for c in c_names], color="#3498DB", alpha=0.85)
for bar, c in zip(bars, c_names):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01, f"{attr_acc[c]:.2f}",
            ha="center", va="bottom", fontsize=12, color="black", fontweight="bold")
ax.axhline(0.25, color="gray", linestyle="--", linewidth=1, label="Random (4 factors)")
ax.set_ylim(0, 1.15); ax.set_xticks(x_pos); ax.set_xticklabels(c_names, rotation=20, ha="right")
ax.set_ylabel("Attribution Accuracy"); ax.set_title("Factor Attribution Accuracy")
ax.legend(fontsize=11); ax.grid(True, alpha=0.2, axis="y")

ax = axes[1]
bars = ax.bar(x_pos, [auroc_sv[c] for c in c_names], color="#E74C3C", alpha=0.85)
for bar, c in zip(bars, c_names):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01, f"{auroc_sv[c]:.3f}",
            ha="center", va="bottom", fontsize=12, color="black", fontweight="bold")
ax.axhline(0.5, color="gray", linestyle="--", linewidth=1, label="Chance (0.5)")
ax.set_ylim(0, 1.15); ax.set_xticks(x_pos); ax.set_xticklabels(c_names, rotation=20, ha="right")
ax.set_ylabel("AUROC (clean vs corrupted)"); ax.set_title("Domain Detection AUROC")
ax.legend(fontsize=11); ax.grid(True, alpha=0.2, axis="y")

fig.suptitle("Synthetic Validation: CD-DSD Attribution Accuracy & Detection AUROC",
             fontsize=15, color="black", fontweight="bold")
plt.tight_layout()
sv_path = os.path.join(SV_DIR, "synthetic_validation.png")
plt.savefig(sv_path, dpi=150, bbox_inches="tight"); plt.close()

print("="*60)
print("  SYNTHETIC VALIDATION RESULTS")
print("="*60)
print(f"  {'Corruption':<14} Attr.Acc   AUROC")
for c in c_names:
    print(f"  {c:<14} {attr_acc[c]:.3f}      {auroc_sv[c]:.3f}")
print(f"  Plot -> {sv_path}")

22:46:28 INFO cd_dsd.datasets: Validating CheXpertDataset...
22:46:29 INFO cd_dsd.datasets:   Valid: 6766/6766 images (100.0%)
22:50:14 INFO synthetic_val:   brightness   attr_acc=0.96  AUROC=1.000
22:53:52 INFO synthetic_val:   darkness     attr_acc=1.00  AUROC=0.988
22:57:29 INFO synthetic_val:   noise        attr_acc=1.00  AUROC=1.000
23:01:06 INFO synthetic_val:   blur         attr_acc=1.00  AUROC=0.952
23:04:44 INFO synthetic_val:   contrast     attr_acc=0.94  AUROC=0.996
23:08:21 INFO synthetic_val:   combined     attr_acc=1.00  AUROC=0.977


  SYNTHETIC VALIDATION RESULTS
  Corruption     Attr.Acc   AUROC
  brightness     0.963      1.000
  darkness       1.000      0.988
  noise          1.000      1.000
  blur           1.000      0.952
  contrast       0.938      0.996
  combined       1.000      0.977
  Plot -> /home/dawood/lab2_rotaion/counterfactual_diff_uncertainty/results/synthetic_validation/synthetic_validation.png


### Baseline Methods for Domain-Shift / OOD Detection (2017–2026)

In [3]:
# BaselineSuite is now in cd_dsd/baselines.py (imported in Quick Setup above).
# ReAct (NeurIPS 2021), DICE (ECCV 2022), ASH (ICLR 2023) added alongside
# MSP, Mahalanobis, Energy, ViM, KNN, GEN, DDA, DiffPath, EigenScore.
#
# suite = BaselineSuite(diagnoser, cfg)   <- already fitted in Quick Setup
print("Baseline methods loaded from cd_dsd.baselines. Run Quick Setup if suite is not defined.")

Baseline methods loaded from cd_dsd.baselines. Run Quick Setup if suite is not defined.


### Run Baseline Comparison vs CD-DSD (AUROC / FPR@95)

In [3]:
"""
Run Baseline Comparison vs CD-DSD (AUROC / FPR@95).
Requires: Quick Setup cell to have been run (diagnoser, suite, cfg, etc.)
Requires: CORRUPTIONS dict from the Synthetic Validation cell (058cda26).
"""
import csv, numpy as np, torch, os
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
from torch.utils.data import DataLoader

RCPARAMS = {"axes.titlesize": 15, "axes.labelsize": 13,
            "xtick.labelsize": 11, "ytick.labelsize": 11,
            "axes.titlecolor": "black", "axes.labelcolor": "black",
            "xtick.color": "black", "ytick.color": "black"}
plt.rcParams.update(RCPARAMS)

COMP_DIR = os.path.join(cfg.RESULTS_DIR, "baseline_comparison")
Path(COMP_DIR).mkdir(parents=True, exist_ok=True)

def fpr_at_95_tpr(clean_s, corrupt_s):
    thr = np.percentile(corrupt_s, 5)
    return float((clean_s >= thr).mean())

eval_tf = get_eval_transform(cfg.IMAGE_SIZE, cfg.MEAN, cfg.STD)
ds = CheXpertDataset(cfg.CHEXPERT_VALID_CSV, cfg.CHEXPERT_IMAGE_ROOT,
                     cfg.LABEL_COLS, eval_tf)
loader = DataLoader(ds, batch_size=8, shuffle=False, collate_fn=collate_fn, num_workers=2)
clean_batches = []
for imgs, _, _ in loader:
    clean_batches.append(imgs)
    if sum(len(b) for b in clean_batches) >= 100: break
clean_images = torch.cat(clean_batches)[:100].to(diagnoser.device)

methods = suite.all_methods()
corruption_names = list(CORRUPTIONS.keys())

# Score clean images once per method
clean_scores = {}
for name, fn in methods.items():
    s = []
    for i in range(0, len(clean_images), 8):
        s.append(fn(clean_images[i:i+8]).cpu().numpy())
    clean_scores[name] = np.concatenate(s)

auroc_table = {n: {} for n in methods}
fpr95_table = {n: {} for n in methods}

for c_name, (c_fn, _) in CORRUPTIONS.items():
    x_c = c_fn(clean_images.clone())
    logger.info(f"Scoring: {c_name}")
    for name, fn in methods.items():
        s = []
        for i in range(0, len(x_c), 8):
            s.append(fn(x_c[i:i+8]).cpu().numpy())
        scores = np.concatenate(s)
        labels = np.concatenate([np.zeros_like(clean_scores[name]),
                                  np.ones_like(scores)])
        all_s = np.concatenate([clean_scores[name], scores])
        try:
            auroc = roc_auc_score(labels, all_s)
        except Exception:
            auroc = float("nan")
        auroc_table[name][c_name] = auroc
        fpr95_table[name][c_name] = fpr_at_95_tpr(clean_scores[name], scores)

# Save CSV
csv_path = os.path.join(COMP_DIR, "baseline_comparison_auroc.csv")
with open(csv_path, "w", newline="") as f:
    w = csv.writer(f); w.writerow(["method"] + corruption_names + ["mean_AUROC"])
    for name in methods:
        row = [auroc_table[name][c] for c in corruption_names]
        w.writerow([name] + [round(v,4) for v in row] + [round(np.nanmean(row),4)])
logger.info(f"CSV -> {csv_path}")

# -- Double-column AUROC heatmap (15" wide) --
method_names = list(methods.keys())
matrix = np.array([[auroc_table[n][c] for c in corruption_names] for n in method_names])
fig, ax = plt.subplots(figsize=(15, 0.7 * len(method_names) + 1.5))
im = ax.imshow(matrix, cmap="RdYlGn", vmin=0.5, vmax=1.0, aspect="auto")
ax.set_xticks(range(len(corruption_names)))
ax.set_xticklabels(corruption_names, rotation=30, ha="right", fontsize=12, color="black")
ax.set_yticks(range(len(method_names)))
ax.set_yticklabels(method_names, fontsize=11, color="black")
for i in range(len(method_names)):
    for j in range(len(corruption_names)):
        val = matrix[i, j]
        ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                fontsize=11, color="black", fontweight="bold")
cbar = plt.colorbar(im, ax=ax)
cbar.set_label("AUROC", fontsize=13, color="black")
cbar.ax.tick_params(labelsize=11, labelcolor="black")
ax.set_title("Baseline Comparison: AUROC per Corruption Type", fontsize=15, color="black")
plt.tight_layout()
hmap_path = os.path.join(COMP_DIR, "baseline_comparison_auroc.png")
plt.savefig(hmap_path, dpi=150, bbox_inches="tight"); plt.close()
logger.info(f"Heatmap -> {hmap_path}")

# Summary
ranked = sorted(method_names, key=lambda n: np.nanmean([auroc_table[n][c] for c in corruption_names]), reverse=True)
print("\n" + "="*65)
print("  BASELINE COMPARISON — mean AUROC across all corruption types")
print("="*65)
for name in ranked:
    mean_a = np.nanmean([auroc_table[name][c] for c in corruption_names])
    print(f"  {name:<25} mean AUROC = {mean_a:.3f}")
print("="*65)

NameError: name 'cfg' is not defined

### JBHI Experiments

#### Exp 1: Downstream Utility — Does U_total Predict Classifier Errors?

In [2]:
"""JBHI Exp 1 — Downstream Utility: does U_total predict classifier errors?"""
import os, numpy as np, pandas as pd, torch, torch.nn.functional as F
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import spearmanr
from torch.utils.data import DataLoader

plt.rcParams.update({"axes.titlesize":15,"axes.labelsize":13,"xtick.labelsize":11,
                     "ytick.labelsize":11,"axes.titlecolor":"black",
                     "axes.labelcolor":"black","xtick.color":"black","ytick.color":"black"})

DU_N, DU_BATCH = 500, 16
DU_DIR = os.path.join(cfg.RESULTS_DIR, "jbhi_downstream_utility")
Path(DU_DIR).mkdir(parents=True, exist_ok=True)

eval_tf  = get_eval_transform(cfg.IMAGE_SIZE, cfg.MEAN, cfg.STD)
mimic_ds = MIMICCXRDataset(cfg.MIMIC_VALID_CSV, cfg.MIMIC_IMAGE_ROOT,
                            cfg.LABEL_COLS, eval_tf, max_samples=DU_N)
loader   = DataLoader(mimic_ds, batch_size=DU_BATCH, shuffle=True, num_workers=4)

rows = []
with torch.no_grad():
    for batch in loader:
        x, labels = batch[0].to(diagnoser.device), batch[1].float()
        results = diagnoser.diagnose_batch(x, domain="mimic", save_vis=False, save_dir=DU_DIR)
        for i, r in enumerate(results):
            pred = torch.tensor(r["pred_original"]).clamp(1e-6, 1-1e-6)
            bce  = F.binary_cross_entropy(pred, labels[i]).item()
            rows.append({"u_total":r["u_total"],"u_domain":r["u_domain"],
                         "u_semantic":r["u_semantic"],"bce_loss":bce})
        if len(rows) >= DU_N: break

df = pd.DataFrame(rows[:DU_N])
rho_t, p_t = spearmanr(df.u_total,    df.bce_loss)
rho_d, p_d = spearmanr(df.u_domain,   df.bce_loss)
rho_s, p_s = spearmanr(df.u_semantic, df.bce_loss)

print("="*60)
print(f"  DOWNSTREAM UTILITY  (N={len(df)} MIMIC-CXR images)")
print("="*60)
print(f"  Spearman rho(U_total,    BCE): {rho_t:+.3f}  p={p_t:.4f}")
print(f"  Spearman rho(U_domain,   BCE): {rho_d:+.3f}  p={p_d:.4f}")
print(f"  Spearman rho(U_semantic, BCE): {rho_s:+.3f}  p={p_s:.4f}")

q25, q50, q75 = np.percentile(df.u_total, [25, 50, 75])
bins = [("Q1 Low", df.u_total<=q25), ("Q2 Med-Low",(df.u_total>q25)&(df.u_total<=q50)),
        ("Q3 Med-High",(df.u_total>q50)&(df.u_total<=q75)), ("Q4 High",df.u_total>q75)]
for name, mask in bins:
    print(f"    {name:<12}: mean BCE={df.bce_loss[mask].mean():.4f}  n={mask.sum()}")
print("="*60)

# -- Double-column figure (14") --
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.scatter(df.u_total, df.bce_loss, alpha=0.3, s=15, color="#2196F3")
ax.set_xlabel("U_total"); ax.set_ylabel("Per-image BCE loss")
ax.set_title(f"U_total vs Classifier Error\nSpearman ρ={rho_t:.3f}, p={p_t:.4f}")
ax.grid(True, alpha=0.2)
# Annotate rho in black
ax.annotate(f"ρ={rho_t:.3f}", xy=(0.05,0.93), xycoords="axes fraction",
            fontsize=12, color="black", fontweight="bold")

ax = axes[1]
means = [df.bce_loss[m].mean() for _, m in bins]
ses   = [df.bce_loss[m].std()/np.sqrt(m.sum()) for _, m in bins]
bars  = ax.bar([n for n,_ in bins], means, yerr=ses, capsize=5,
               color=["#4CAF50","#FFC107","#FF9800","#F44336"], alpha=0.85)
for bar, val in zip(bars, means):
    ax.text(bar.get_x()+bar.get_width()/2, val+max(ses)*0.3,
            f"{val:.3f}", ha="center", va="bottom", fontsize=11, color="black", fontweight="bold")
ax.set_xlabel("U_total Quartile"); ax.set_ylabel("Mean BCE ± SE")
ax.set_title("Classifier Error by Uncertainty Quartile")
ax.grid(True, alpha=0.2, axis="y")

fig.suptitle("Downstream Utility: U_total Predicts Classifier Error on MIMIC-CXR",
             fontsize=15, color="black", fontweight="bold")
plt.tight_layout()
plot_path = os.path.join(DU_DIR, "downstream_utility.png")
plt.savefig(plot_path, dpi=150, bbox_inches="tight"); plt.close()
df.to_csv(os.path.join(DU_DIR, "downstream_utility.csv"), index=False)
print(f"  Plot -> {plot_path}")

NameError: name 'cfg' is not defined

#### Exp 2: Ablation Study — Components and Hyperparameters

In [6]:
"""JBHI Exp 2 — Ablation: components (U_domain+HFER vs U_semantic vs U_total)
and hyperparameters (n_timesteps, mc_samples)."""
import os, numpy as np, torch
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader

plt.rcParams.update({"axes.titlesize":15,"axes.labelsize":13,"xtick.labelsize":11,
                     "ytick.labelsize":11,"axes.titlecolor":"black",
                     "axes.labelcolor":"black","xtick.color":"black","ytick.color":"black"})

ABL_N, ABL_BATCH = 100, 16
ABL_DIR = os.path.join(cfg.RESULTS_DIR, "jbhi_ablation")
Path(ABL_DIR).mkdir(parents=True, exist_ok=True)

eval_tf = get_eval_transform(cfg.IMAGE_SIZE, cfg.MEAN, cfg.STD)
chex_ds = CheXpertDataset(cfg.CHEXPERT_VALID_CSV, cfg.CHEXPERT_IMAGE_ROOT,
                           cfg.LABEL_COLS, eval_tf)
loader  = DataLoader(chex_ds, batch_size=ABL_BATCH, shuffle=True, num_workers=4)
device  = diagnoser.device

def collect_signals(n_samples, corr_fn=None):
    ud_list, us_list, ut_list = [], [], []
    n = 0
    for batch in loader:
        if n >= n_samples: break
        x = batch[0].to(device)
        if corr_fn: x = corr_fn(x)
        with torch.no_grad():
            # U_domain: full combined (diffusion + HFER) via suite
            ud = suite.score_cd_dsd(x).cpu().numpy()
            us = diagnoser.classifier.uncertainty_scalar(
                     x, n_samples=cfg.MC_SAMPLES).cpu().numpy()
        ut_list.append(ud + us)
        ud_list.append(ud); us_list.append(us)
        n += x.shape[0]
    return (np.concatenate(ud_list)[:n_samples],
            np.concatenate(us_list)[:n_samples],
            np.concatenate(ut_list)[:n_samples])

corr_fns = {"brightness":corrupt_brightness,"darkness":corrupt_darkness,
            "noise":corrupt_noise,"blur":corrupt_blur,"contrast":corrupt_contrast}

# --- Part A: Component ablation ---
ud_clean, us_clean, ut_clean = collect_signals(ABL_N)
auroc_domain, auroc_sem, auroc_total = {}, {}, {}
for cname, cfn in corr_fns.items():
    ud_c, us_c, ut_c = collect_signals(ABL_N, cfn)
    lbl = np.array([0]*ABL_N + [1]*ABL_N)
    for scores, store in [(np.concatenate([ud_clean,ud_c]), auroc_domain),
                           (np.concatenate([us_clean,us_c]), auroc_sem),
                           (np.concatenate([ut_clean,ut_c]), auroc_total)]:
        try:    store[cname] = roc_auc_score(lbl, scores)
        except: store[cname] = float("nan")

labels_str = list(corr_fns.keys())
x_pos  = np.arange(len(labels_str))
width  = 0.25

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

ax = axes[0]
b1 = ax.bar(x_pos - width, [auroc_domain[c] for c in labels_str], width, label="U_domain+HFER", color="#E74C3C", alpha=0.85)
b2 = ax.bar(x_pos,         [auroc_sem[c]    for c in labels_str], width, label="U_semantic",    color="#3498DB", alpha=0.85)
b3 = ax.bar(x_pos + width, [auroc_total[c]  for c in labels_str], width, label="U_total",       color="#2ECC71", alpha=0.85)
for bars in [b1, b2, b3]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x()+bar.get_width()/2, h+0.005, f"{h:.2f}",
                ha="center", va="bottom", fontsize=9, color="black")
ax.axhline(0.5, color="gray", linestyle="--", linewidth=1)
ax.set_ylim(0, 1.1); ax.set_xticks(x_pos); ax.set_xticklabels(labels_str, rotation=20, ha="right")
ax.set_ylabel("AUROC"); ax.set_title("Part A: Component Ablation")
ax.legend(fontsize=10); ax.grid(True, alpha=0.2, axis="y")

# --- Part B: Hyperparameter ablation (n_timesteps) ---
ts_vals = [5, 10, 20]
ax = axes[1]
auroc_ts = []
ud_clean_base, _, _ = collect_signals(ABL_N)
ud_c_base, _, _     = collect_signals(ABL_N, corrupt_noise)
lbl = np.array([0]*ABL_N + [1]*ABL_N)
for ts in ts_vals:
    orig_ts = cfg.DOMAIN_SCORE_TIMESTEPS
    cfg.DOMAIN_SCORE_TIMESTEPS = ts
    ud_cl, _, _ = collect_signals(ABL_N)
    ud_cr, _, _ = collect_signals(ABL_N, corrupt_noise)
    cfg.DOMAIN_SCORE_TIMESTEPS = orig_ts
    try:    auroc_ts.append(roc_auc_score(lbl, np.concatenate([ud_cl, ud_cr])))
    except: auroc_ts.append(float("nan"))
bars = ax.bar([str(t) for t in ts_vals], auroc_ts, color=["#9B59B6","#E67E22","#1ABC9C"], alpha=0.85)
for bar, val in zip(bars, auroc_ts):
    ax.text(bar.get_x()+bar.get_width()/2, val+0.01, f"{val:.3f}",
            ha="center", va="bottom", fontsize=12, color="black", fontweight="bold")
ax.set_ylim(0, 1.1); ax.set_xlabel("n_timesteps"); ax.set_ylabel("AUROC (noise)")
ax.set_title("Part B: Hyperparameter Ablation"); ax.grid(True, alpha=0.2, axis="y")

fig.suptitle("Ablation Study — CD-DSD Components & Hyperparameters",
             fontsize=15, color="black", fontweight="bold")
plt.tight_layout()
abl_path = os.path.join(ABL_DIR, "ablation_study.png")
plt.savefig(abl_path, dpi=150, bbox_inches="tight"); plt.close()
print(f"  Plot -> {abl_path}")

23:35:21 INFO cd_dsd.datasets: Validating CheXpertDataset...
23:35:22 INFO cd_dsd.datasets:   Valid: 6766/6766 images (100.0%)
23:45:47 INFO matplotlib.category: Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
23:45:47 INFO matplotlib.category: Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.


  Plot -> /home/dawood/lab2_rotaion/counterfactual_diff_uncertainty/results/jbhi_ablation/ablation_study.png


#### Exp 3: Bootstrap Confidence Intervals on AUROC

In [7]:
"""JBHI Exp 3 — Bootstrap 95% CI on AUROC (1000 resamples)."""
import os, numpy as np, torch
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader

plt.rcParams.update({"axes.titlesize":15,"axes.labelsize":13,"xtick.labelsize":11,
                     "ytick.labelsize":11,"axes.titlecolor":"black",
                     "axes.labelcolor":"black","xtick.color":"black","ytick.color":"black"})

N_BOOT, BS_N = 1000, 100
BS_DIR = os.path.join(cfg.RESULTS_DIR, "jbhi_bootstrap_ci")
Path(BS_DIR).mkdir(parents=True, exist_ok=True)

eval_tf = get_eval_transform(cfg.IMAGE_SIZE, cfg.MEAN, cfg.STD)
chex_ds = CheXpertDataset(cfg.CHEXPERT_VALID_CSV, cfg.CHEXPERT_IMAGE_ROOT,
                           cfg.LABEL_COLS, eval_tf)
loader  = DataLoader(chex_ds, batch_size=16, shuffle=True, num_workers=4)
device  = diagnoser.device

corr_fns = {"brightness":corrupt_brightness,"darkness":corrupt_darkness,
            "noise":corrupt_noise,"blur":corrupt_blur,"contrast":corrupt_contrast}

# Methods to compare with Bootstrap CI
bs_methods = {
    "CD-DSD": suite.score_cd_dsd,
    "ViM":    suite.score_vim,
    "KNN":    suite.score_knn,
    "MSP":    suite.score_msp,
}

def score_batches(fn, images):
    s = []
    for i in range(0, len(images), 16):
        s.append(fn(images[i:i+16]).cpu().numpy())
    return np.concatenate(s)

# Collect all images
clean_batches = []
for batch in loader:
    clean_batches.append(batch[0])
    if sum(len(b) for b in clean_batches) >= BS_N: break
clean_images = torch.cat(clean_batches)[:BS_N].to(device)

# Per-corruption Bootstrap CI
results = {n: {} for n in bs_methods}
for cname, cfn in corr_fns.items():
    x_c = cfn(clean_images.clone())
    for mname, mfn in bs_methods.items():
        clean_s  = score_batches(mfn, clean_images)
        corr_s   = score_batches(mfn, x_c)
        all_s    = np.concatenate([clean_s, corr_s])
        lbl      = np.array([0]*BS_N + [1]*BS_N)
        boot     = []
        for _ in range(N_BOOT):
            idx = np.random.choice(len(lbl), len(lbl), replace=True)
            try:    boot.append(roc_auc_score(lbl[idx], all_s[idx]))
            except: pass
        boot = np.array(boot)
        results[mname][cname] = (boot.mean(), np.percentile(boot,2.5), np.percentile(boot,97.5))

# -- Double-column plot (14") --
x_pos   = np.arange(len(corr_fns))
cnames  = list(corr_fns.keys())
colors  = {"CD-DSD":"#E74C3C","ViM":"#3498DB","KNN":"#2ECC71","MSP":"#F39C12"}
width   = 0.18
offsets = np.linspace(-1.5*width, 1.5*width, len(bs_methods))

fig, ax = plt.subplots(figsize=(14, 5))
for (mname, _), offset in zip(bs_methods.items(), offsets):
    means = [results[mname][c][0] for c in cnames]
    lows  = [results[mname][c][0]-results[mname][c][1] for c in cnames]
    highs = [results[mname][c][2]-results[mname][c][0] for c in cnames]
    bars  = ax.bar(x_pos+offset, means, width, yerr=[lows,highs], capsize=4,
                   label=mname, color=colors[mname], alpha=0.85, error_kw={"elinewidth":1.5,"ecolor":"black"})
    for bar, val in zip(bars, means):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+max(highs)*0.3,
                f"{val:.2f}", ha="center", va="bottom", fontsize=8, color="black")
ax.axhline(0.5, color="gray", linestyle="--", linewidth=1)
ax.set_ylim(0, 1.15)
ax.set_xticks(x_pos); ax.set_xticklabels(cnames, rotation=15, ha="right")
ax.set_ylabel("AUROC"); ax.legend(fontsize=11)
ax.set_title("Bootstrap 95% CI on AUROC — CD-DSD vs Top Baselines", fontsize=15, color="black")
ax.grid(True, alpha=0.2, axis="y")
plt.tight_layout()
bs_path = os.path.join(BS_DIR, "bootstrap_ci_auroc.png")
plt.savefig(bs_path, dpi=150, bbox_inches="tight"); plt.close()

print("="*65)
print("  BOOTSTRAP AUROC (mean [95% CI])")
print("="*65)
for mname in bs_methods:
    vals = [results[mname][c] for c in cnames]
    overall_mean = np.mean([v[0] for v in vals])
    print(f"  {mname:<10} overall={overall_mean:.3f}")
    for c, (m, lo, hi) in zip(cnames, vals):
        print(f"    {c:<14} {m:.3f} [{lo:.3f}, {hi:.3f}]")
print(f"  Plot -> {bs_path}")

23:45:48 INFO cd_dsd.datasets: Validating CheXpertDataset...
23:45:49 INFO cd_dsd.datasets:   Valid: 6766/6766 images (100.0%)


  BOOTSTRAP AUROC (mean [95% CI])
  CD-DSD     overall=0.997
    brightness     0.997 [0.990, 1.000]
    darkness       0.997 [0.989, 1.000]
    noise          1.000 [1.000, 1.000]
    blur           0.992 [0.973, 1.000]
    contrast       1.000 [1.000, 1.000]
  ViM        overall=0.834
    brightness     0.610 [0.527, 0.685]
    darkness       0.814 [0.749, 0.873]
    noise          1.000 [1.000, 1.000]
    blur           0.959 [0.934, 0.981]
    contrast       0.786 [0.715, 0.850]
  KNN        overall=0.850
    brightness     0.541 [0.459, 0.619]
    darkness       0.883 [0.835, 0.924]
    noise          1.000 [1.000, 1.000]
    blur           0.991 [0.982, 0.998]
    contrast       0.836 [0.781, 0.886]
  MSP        overall=0.600
    brightness     0.578 [0.498, 0.660]
    darkness       0.685 [0.610, 0.762]
    noise          0.419 [0.340, 0.502]
    blur           0.661 [0.581, 0.737]
    contrast       0.657 [0.579, 0.729]
  Plot -> /home/dawood/lab2_rotaion/counterfactual_diff_un

#### Exp 4: Real-Domain Attribution — What Factor Does Each Target Domain Exhibit?

In [8]:
"""JBHI Exp 4 — Real-Domain Attribution: MIMIC-CXR vs NIH-ChestXray14."""
import os, numpy as np
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
from torch.utils.data import DataLoader

plt.rcParams.update({"axes.titlesize":15,"axes.labelsize":13,"xtick.labelsize":11,
                     "ytick.labelsize":11,"axes.titlecolor":"black",
                     "axes.labelcolor":"black","xtick.color":"black","ytick.color":"black"})

ATTR_N, ATTR_BATCH = 200, 16
ATTR_DIR = os.path.join(cfg.RESULTS_DIR, "jbhi_attribution_analysis")
Path(ATTR_DIR).mkdir(parents=True, exist_ok=True)

eval_tf = get_eval_transform(cfg.IMAGE_SIZE, cfg.MEAN, cfg.STD)
factor_keys = sorted(cfg.T_STAR_LEVELS.keys())

def run_attribution(ds, domain_name, n):
    loader = DataLoader(ds, batch_size=ATTR_BATCH, shuffle=True, num_workers=4)
    top_factors, u_totals, u_domains = [], [], []
    n_done = 0
    for batch in loader:
        if n_done >= n: break
        x = batch[0].to(diagnoser.device)
        results = diagnoser.diagnose_batch(x, domain=domain_name, save_vis=False, save_dir=ATTR_DIR)
        for r in results:
            top = max(r["factor_attributions"], key=r["factor_attributions"].get)
            top_factors.append(top); u_totals.append(r["u_total"]); u_domains.append(r["u_domain"])
        n_done += len(results)
    return top_factors[:n], np.array(u_totals[:n]), np.array(u_domains[:n])

mimic_ds = MIMICCXRDataset(cfg.MIMIC_VALID_CSV, cfg.MIMIC_IMAGE_ROOT,
                            cfg.LABEL_COLS, eval_tf, max_samples=ATTR_N)
nih_ds   = NIHChestXrayDataset(cfg.NIH_CSV, cfg.NIH_IMAGE_ROOT,
                                cfg.LABEL_COLS, eval_tf, max_samples=ATTR_N)
logger.info("Running MIMIC-CXR attribution...")
mimic_factors, mimic_ut, mimic_ud = run_attribution(mimic_ds, "MIMIC-CXR", ATTR_N)
logger.info("Running NIH-ChestXray14 attribution...")
nih_factors,   nih_ut,   nih_ud   = run_attribution(nih_ds,   "NIH-Xray14",  ATTR_N)

# Factor frequency per domain
def factor_dist(factors):
    counts = {k: 0 for k in factor_keys}
    for f in factors: counts[f] = counts.get(f, 0) + 1
    total = len(factors)
    return [counts.get(k, 0)/total for k in factor_keys]

mimic_dist = factor_dist(mimic_factors)
nih_dist   = factor_dist(nih_factors)

# -- Single-column figure (10") --
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
x_pos = np.arange(len(factor_keys))
width = 0.35

ax = axes[0]
bars_m = ax.bar(x_pos - width/2, mimic_dist, width, label="MIMIC-CXR",   color="#3498DB", alpha=0.85)
bars_n = ax.bar(x_pos + width/2, nih_dist,   width, label="NIH-Xray14",  color="#E74C3C", alpha=0.85)
for bars, vals in [(bars_m, mimic_dist), (bars_n, nih_dist)]:
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, v+0.01, f"{v:.2f}",
                ha="center", va="bottom", fontsize=10, color="black")
ax.set_xticks(x_pos); ax.set_xticklabels(factor_keys, rotation=20, ha="right")
ax.set_ylabel("Fraction of images"); ax.set_title("Top Factor Attribution by Domain")
ax.legend(fontsize=11); ax.grid(True, alpha=0.2, axis="y")

ax = axes[1]
for domain, ut, ud, color in [("MIMIC", mimic_ut, mimic_ud, "#3498DB"),
                                ("NIH",   nih_ut,   nih_ud,   "#E74C3C")]:
    ax.scatter(ud, ut - ud, alpha=0.3, s=15, color=color, label=domain)
ax.set_xlabel("U_domain (HFER+diffusion)"); ax.set_ylabel("U_semantic")
ax.set_title("Domain vs Semantic Uncertainty")
ax.legend(fontsize=11); ax.grid(True, alpha=0.2)

fig.suptitle("Real-Domain Attribution: MIMIC-CXR vs NIH-ChestXray14",
             fontsize=15, color="black", fontweight="bold")
plt.tight_layout()
attr_path = os.path.join(ATTR_DIR, "real_attribution.png")
plt.savefig(attr_path, dpi=150, bbox_inches="tight"); plt.close()
print(f"  Plot -> {attr_path}")
print(f"  MIMIC top factor: {max(set(mimic_factors), key=mimic_factors.count)}")
print(f"  NIH   top factor: {max(set(nih_factors),   key=nih_factors.count)}")

23:51:47 INFO cd_dsd.datasets: Validating MIMICCXRDataset...
23:51:47 INFO cd_dsd.datasets:   Valid: 200/200 images (100.0%)
23:51:47 INFO cd_dsd.datasets: Validating NIHChestXrayDataset...
23:51:50 INFO cd_dsd.datasets:   Valid: 200/200 images (100.0%)
23:51:50 INFO synthetic_val: Running MIMIC-CXR attribution...
23:58:28 INFO synthetic_val: Running NIH-ChestXray14 attribution...


  Plot -> /home/dawood/lab2_rotaion/counterfactual_diff_uncertainty/results/jbhi_attribution_analysis/real_attribution.png
  MIMIC top factor: global_structure
  NIH   top factor: global_structure


#### Exp 5: Qualitative Visualization — High / Medium / Low Uncertainty Examples

In [9]:
"""JBHI Exp 5 — Qualitative: High/Medium/Low uncertainty examples from MIMIC-CXR."""
import os, numpy as np, torch
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
from torch.utils.data import DataLoader

plt.rcParams.update({"axes.titlesize":14,"axes.labelsize":12,"xtick.labelsize":10,
                     "ytick.labelsize":10,"axes.titlecolor":"black","axes.labelcolor":"black"})

VIS_N, VIS_DIR = 80, os.path.join(cfg.RESULTS_DIR, "jbhi_qualitative")
Path(VIS_DIR).mkdir(parents=True, exist_ok=True)

eval_tf  = get_eval_transform(cfg.IMAGE_SIZE, cfg.MEAN, cfg.STD)
mimic_ds = MIMICCXRDataset(cfg.MIMIC_VALID_CSV, cfg.MIMIC_IMAGE_ROOT,
                            cfg.LABEL_COLS, eval_tf, max_samples=VIS_N)
loader   = DataLoader(mimic_ds, batch_size=16, shuffle=True, num_workers=4)
device   = diagnoser.device

mean_t = torch.tensor(cfg.MEAN).view(-1,1,1)
std_t  = torch.tensor(cfg.STD).view(-1,1,1)

def to_gray(x):
    img = (x.cpu() * std_t + mean_t).clamp(0,1)
    return img.mean(0).numpy()

all_imgs, all_results = [], []
with torch.no_grad():
    for batch in loader:
        x = batch[0].to(device)
        results = diagnoser.diagnose_batch(x, domain="mimic", save_vis=False, save_dir=VIS_DIR)
        for i, r in enumerate(results):
            all_imgs.append(x[i].cpu()); all_results.append(r)
        if len(all_imgs) >= VIS_N: break

u_vals     = np.array([r["u_total"] for r in all_results])
sorted_idx = np.argsort(u_vals)
n = len(sorted_idx)
tiers = {"High": sorted_idx[-3:], "Medium": sorted_idx[n//2-1:n//2+2], "Low": sorted_idx[:3]}

# -- Double-column figure (14") --
n_cols = 3
fig, axes = plt.subplots(len(tiers)*2, n_cols, figsize=(14, len(tiers)*4.5))

for row_i, (tier, idxs) in enumerate(tiers.items()):
    for col_i, si in enumerate(idxs[:n_cols]):
        r   = all_results[si]
        img = all_imgs[si]
        top = max(r["factor_attributions"], key=r["factor_attributions"].get)

        with torch.no_grad():
            x_star = diagnoser.diffusion.partial_correct(
                img.unsqueeze(0).to(device), cfg.T_STAR_FULL,
                num_steps=cfg.DDIM_STEPS, eta=cfg.DDIM_ETA)[0]

        title_orig = (tier + " U | Ud=" + f"{r['u_domain']:.2f}"
                      + " Us=" + f"{r['u_semantic']:.2f}" + " | " + top)

        ax = axes[row_i*2, col_i]
        ax.imshow(to_gray(img), cmap="gray", vmin=0, vmax=1)
        ax.set_title(title_orig, fontsize=10, color="black")
        ax.axis("off")

        ax = axes[row_i*2+1, col_i]
        ax.imshow(to_gray(x_star), cmap="gray", vmin=0, vmax=1)
        ax.set_title("Corrected (x*)", fontsize=10, color="black")
        ax.axis("off")

fig.suptitle("CD-DSD Qualitative: High / Medium / Low Uncertainty Examples (MIMIC-CXR)",
             fontsize=14, color="black", fontweight="bold")
plt.tight_layout()
vis_path = os.path.join(VIS_DIR, "qualitative_examples.png")
plt.savefig(vis_path, dpi=150, bbox_inches="tight"); plt.close()
print("Plot ->", vis_path)

00:05:14 INFO cd_dsd.datasets: Validating MIMICCXRDataset...
00:05:14 INFO cd_dsd.datasets:   Valid: 80/80 images (100.0%)


Plot -> /home/dawood/lab2_rotaion/counterfactual_diff_uncertainty/results/jbhi_qualitative/qualitative_examples.png


#### Exp 6: Computational Cost — Inference Time per Image

In [10]:
"""JBHI Exp 6 — Computational Cost: inference time (ms/image)."""
import os, time, numpy as np, torch
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
from torch.utils.data import DataLoader

plt.rcParams.update({"axes.titlesize":15,"axes.labelsize":13,"xtick.labelsize":11,
                     "ytick.labelsize":11,"axes.titlecolor":"black",
                     "axes.labelcolor":"black","xtick.color":"black","ytick.color":"black"})

COST_DIR = os.path.join(cfg.RESULTS_DIR, "jbhi_compute_cost")
Path(COST_DIR).mkdir(parents=True, exist_ok=True)

eval_tf = get_eval_transform(cfg.IMAGE_SIZE, cfg.MEAN, cfg.STD)
chex_ds = CheXpertDataset(cfg.CHEXPERT_VALID_CSV, cfg.CHEXPERT_IMAGE_ROOT, cfg.LABEL_COLS, eval_tf)
loader  = DataLoader(chex_ds, batch_size=16, shuffle=True, num_workers=4)
device  = diagnoser.device
batch_x = next(iter(loader))[0].to(device)

def time_fn(fn, warmup=3, runs=10):
    for _ in range(warmup): fn()
    if "cuda" in str(device): torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(runs): fn()
    if "cuda" in str(device): torch.cuda.synchronize()
    return (time.perf_counter() - t0) / runs / batch_x.shape[0] * 1000

with torch.no_grad():
    timings = {
        "Classifier backbone":       time_fn(lambda: diagnoser.classifier.predict(batch_x)),
        "MC-Dropout (U_semantic)":   time_fn(lambda: diagnoser.classifier.uncertainty_scalar(batch_x, n_samples=cfg.MC_SAMPLES)),
        "HFER (blur signal)":        time_fn(lambda: hf_energy_score(batch_x, cfg.HFER_RADIUS_FRAC)),
        "Diffusion domain_score":    time_fn(lambda: diagnoser.diffusion.domain_score(batch_x, n_timesteps=cfg.DOMAIN_SCORE_TIMESTEPS)),
        "CD-DSD full (U_total)":     time_fn(lambda: diagnoser.diagnose_batch(batch_x, save_vis=False, save_dir=COST_DIR)),
        "ViM":                        time_fn(lambda: suite.score_vim(batch_x)),
        "KNN":                        time_fn(lambda: suite.score_knn(batch_x)),
        "ReAct":                      time_fn(lambda: suite.score_react(batch_x)),
        "MSP":                        time_fn(lambda: suite.score_msp(batch_x)),
    }

names = list(timings.keys()); vals = [timings[n] for n in names]

fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#E74C3C" if "CD-DSD" in n else "#3498DB" for n in names]
bars = ax.barh(names, vals, color=colors, alpha=0.85)
for bar, val in zip(bars, vals):
    ax.text(val + max(vals)*0.01, bar.get_y()+bar.get_height()/2,
            f"{val:.1f} ms", va="center", ha="left", fontsize=11, color="black", fontweight="bold")
ax.set_xlabel("Inference time (ms / image)"); ax.set_title("Computational Cost: Inference Time per Image")
ax.grid(True, alpha=0.2, axis="x")
plt.tight_layout()
cost_path = os.path.join(COST_DIR, "compute_cost.png")
plt.savefig(cost_path, dpi=150, bbox_inches="tight"); plt.close()

print("="*55)
print("  INFERENCE TIME (ms per image)")
print("="*55)
for n, v in sorted(timings.items(), key=lambda x: x[1]):
    print(f"  {n:<30} {v:.2f} ms")
print(f"  Plot -> {cost_path}")

00:08:14 INFO cd_dsd.datasets: Validating CheXpertDataset...
00:08:15 INFO cd_dsd.datasets:   Valid: 6766/6766 images (100.0%)


  INFERENCE TIME (ms per image)
  HFER (blur signal)             0.05 ms
  Classifier backbone            1.19 ms
  ReAct                          1.35 ms
  KNN                            1.36 ms
  MSP                            1.39 ms
  ViM                            2.73 ms
  MC-Dropout (U_semantic)        33.76 ms
  Diffusion domain_score         82.07 ms
  CD-DSD full (U_total)          2144.30 ms
  Plot -> /home/dawood/lab2_rotaion/counterfactual_diff_uncertainty/results/jbhi_compute_cost/compute_cost.png


#### Exp 7: Disease-Specific Uncertainty — U_domain per Pathology Class

In [11]:
"""JBHI Exp 7 — Disease-Specific Uncertainty: U_domain / U_semantic per pathology."""
import os, numpy as np, pandas as pd, torch, torch.nn.functional as F
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import spearmanr
from torch.utils.data import DataLoader

plt.rcParams.update({"axes.titlesize":15,"axes.labelsize":13,"xtick.labelsize":10,
                     "ytick.labelsize":11,"axes.titlecolor":"black",
                     "axes.labelcolor":"black","xtick.color":"black","ytick.color":"black"})

DS_N, DS_BATCH = 500, 16
DS_DIR = os.path.join(cfg.RESULTS_DIR, "jbhi_disease_specific")
Path(DS_DIR).mkdir(parents=True, exist_ok=True)

eval_tf  = get_eval_transform(cfg.IMAGE_SIZE, cfg.MEAN, cfg.STD)
mimic_ds = MIMICCXRDataset(cfg.MIMIC_VALID_CSV, cfg.MIMIC_IMAGE_ROOT,
                            cfg.LABEL_COLS, eval_tf, max_samples=DS_N)
loader   = DataLoader(mimic_ds, batch_size=DS_BATCH, shuffle=True, num_workers=4)

rows = []
with torch.no_grad():
    for batch in loader:
        if len(rows) >= DS_N: break
        x, labels = batch[0].to(diagnoser.device), batch[1].float()
        results = diagnoser.diagnose_batch(x, domain="mimic", save_vis=False, save_dir=DS_DIR)
        for i, r in enumerate(results):
            pred = torch.tensor(r["pred_original"]).clamp(1e-6,1-1e-6)
            bce  = F.binary_cross_entropy(pred, labels[i]).item()
            row  = {"u_total":r["u_total"],"u_domain":r["u_domain"],"u_semantic":r["u_semantic"],"bce_loss":bce}
            for j, col in enumerate(cfg.LABEL_COLS): row[col] = labels[i][j].item()
            rows.append(row)

df = pd.DataFrame(rows[:DS_N])

mean_ud = {}; mean_us = {}; mean_bce = {}
for cls in cfg.LABEL_COLS:
    pos = df[df[cls] > 0.5]
    if len(pos) < 5: continue
    mean_ud[cls]  = pos.u_domain.mean()
    mean_us[cls]  = pos.u_semantic.mean()
    mean_bce[cls] = pos.bce_loss.mean()

cls_names = list(mean_ud.keys())
x_pos     = np.arange(len(cls_names))
width     = 0.35

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

ax = axes[0]
b1 = ax.bar(x_pos-width/2, [mean_ud[c] for c in cls_names], width, label="U_domain+HFER", color="#E74C3C", alpha=0.85)
b2 = ax.bar(x_pos+width/2, [mean_us[c] for c in cls_names], width, label="U_semantic",    color="#3498DB", alpha=0.85)
for bars in [b1, b2]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x()+bar.get_width()/2, h+0.001, f"{h:.3f}",
                ha="center", va="bottom", fontsize=8, color="black")
ax.set_xticks(x_pos); ax.set_xticklabels(cls_names, rotation=30, ha="right")
ax.set_ylabel("Mean uncertainty"); ax.set_title("Uncertainty by Pathology Class")
ax.legend(fontsize=10); ax.grid(True, alpha=0.2, axis="y")

ax = axes[1]
bars = ax.bar(cls_names, [mean_bce[c] for c in cls_names], color="#9B59B6", alpha=0.85)
for bar, cls in zip(bars, cls_names):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.001, f"{mean_bce[cls]:.3f}",
            ha="center", va="bottom", fontsize=9, color="black")
ax.set_xticks(range(len(cls_names))); ax.set_xticklabels(cls_names, rotation=30, ha="right")
ax.set_ylabel("Mean BCE loss"); ax.set_title("Classifier Error by Pathology Class")
ax.grid(True, alpha=0.2, axis="y")

fig.suptitle("Disease-Specific Uncertainty Analysis (MIMIC-CXR, N=500)",
             fontsize=15, color="black", fontweight="bold")
plt.tight_layout()
ds_path = os.path.join(DS_DIR, "disease_specific_uncertainty.png")
plt.savefig(ds_path, dpi=150, bbox_inches="tight"); plt.close()

print("="*68)
print("  DISEASE-SPECIFIC UNCERTAINTY  (positive-class images only)")
print("="*68)
for cls in cls_names:
    n_pos = int((df[cls]>0.5).sum())
    print(f"  {cls:<20} U_dom={mean_ud[cls]:.4f}  U_sem={mean_us[cls]:.4f}  BCE={mean_bce[cls]:.4f}  n={n_pos}")
print(f"  Plot -> {ds_path}")

00:16:11 INFO cd_dsd.datasets: Validating MIMICCXRDataset...
00:16:11 INFO cd_dsd.datasets:   Valid: 500/500 images (100.0%)


  DISEASE-SPECIFIC UNCERTAINTY  (positive-class images only)
  No Finding           U_dom=0.0750  U_sem=0.2442  BCE=0.5581  n=249
  Atelectasis          U_dom=0.1043  U_sem=0.3167  BCE=0.6439  n=55
  Cardiomegaly         U_dom=0.1023  U_sem=0.3083  BCE=0.7874  n=50
  Consolidation        U_dom=0.1159  U_sem=0.3173  BCE=0.5356  n=7
  Edema                U_dom=0.0791  U_sem=0.3218  BCE=0.7643  n=43
  Pleural Effusion     U_dom=0.1002  U_sem=0.3249  BCE=0.6842  n=65
  Pneumonia            U_dom=0.0700  U_sem=0.3082  BCE=0.8362  n=31
  Pneumothorax         U_dom=0.1319  U_sem=0.3306  BCE=1.1974  n=15
  Plot -> /home/dawood/lab2_rotaion/counterfactual_diff_uncertainty/results/jbhi_disease_specific/disease_specific_uncertainty.png


#### Exp 8: Diffusion Model Fidelity — FID + Disease AUC on Generated Images

Validates that the DDPM generates clinically plausible chest X-rays:
- **Domain FID**: Fréchet distance between CheXpert real features and generated image features (using DenseNet-121 backbone, lower = better)
- **Disease AUC**: run the trained classifier on generated images and compare mean disease probabilities with real images
- This addresses the reviewer concern: "Are your generated/corrected images actually valid CXRs?"

In [6]:
"""JBHI Exp 8 — Diffusion Fidelity: Domain FID + Disease AUC on generated images."""
import os, numpy as np, torch
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.linalg import sqrtm
from torch.utils.data import DataLoader

plt.rcParams.update({"axes.titlesize":15,"axes.labelsize":13,"xtick.labelsize":11,
                     "ytick.labelsize":11,"axes.titlecolor":"black",
                     "axes.labelcolor":"black","xtick.color":"black","ytick.color":"black"})

FID_N   = 1000
FID_DIR = os.path.join(cfg.RESULTS_DIR, "jbhi_fidelity")
Path(FID_DIR).mkdir(parents=True, exist_ok=True)

device = diagnoser.device
diff   = diagnoser.diffusion

# ---- Generate images via DDIM starting from pure noise ----
logger.info(f"Generating {FID_N} images via DDIM ({cfg.DDIM_STEPS} steps)...")
gen_imgs = []
with torch.no_grad():
    for i in range(0, FID_N, 16):
        n_batch = min(16, FID_N - i)
        noise   = torch.randn(n_batch, 3, cfg.IMAGE_SIZE, cfg.IMAGE_SIZE, device=device)
        x_gen   = diff.ddim_sample(noise, num_steps=cfg.DDIM_STEPS, eta=cfg.DDIM_ETA)
        gen_imgs.append(x_gen.cpu())
        logger.info(f"  Generated {i+n_batch}/{FID_N}")
gen_imgs = torch.cat(gen_imgs)

# ---- Load real CheXpert images ----
eval_tf  = get_eval_transform(cfg.IMAGE_SIZE, cfg.MEAN, cfg.STD)
real_ds  = CheXpertDataset(cfg.CHEXPERT_VALID_CSV, cfg.CHEXPERT_IMAGE_ROOT,
                            cfg.LABEL_COLS, eval_tf)
real_loader = DataLoader(real_ds, batch_size=16, shuffle=True,
                          collate_fn=collate_fn, num_workers=2)
real_imgs, real_labs = [], []
for imgs, labs, _ in real_loader:
    real_imgs.append(imgs); real_labs.append(labs)
    if sum(len(b) for b in real_imgs) >= FID_N: break
real_imgs = torch.cat(real_imgs)[:FID_N]
real_labs = torch.cat(real_labs)[:FID_N]

# ---- Extract DenseNet-121 features ----
logger.info("Extracting features for Domain FID...")
def extract_feats(imgs):
    feats = []
    for i in range(0, len(imgs), 16):
        batch = imgs[i:i+16].to(device)
        with torch.no_grad():
            f = diagnoser._extract_features(batch).cpu().numpy()
        feats.append(f)
    return np.concatenate(feats)

real_feats = extract_feats(real_imgs)  # (N, 1024)
gen_feats  = extract_feats(gen_imgs)   # (N, 1024)

# ---- Compute Domain FID via scipy sqrtm (always >= 0) ----
# N=200 < D=1024 -> covariances are rank-deficient -> add eps to diagonal
def frechet_distance(feat_a, feat_b, eps=1e-5):
    mu_a, mu_b = feat_a.mean(0), feat_b.mean(0)
    cov_a = np.cov(feat_a, rowvar=False) + eps * np.eye(feat_a.shape[1])
    cov_b = np.cov(feat_b, rowvar=False) + eps * np.eye(feat_b.shape[1])
    diff  = mu_a - mu_b
    covmean, _ = sqrtm(cov_a @ cov_b, disp=False)
    if np.iscomplexobj(covmean):
        covmean = covmean.real  # imaginary part = floating-point noise only
    fid = float(diff @ diff + np.trace(cov_a + cov_b - 2.0 * covmean))
    return max(0.0, fid)

logger.info("Computing Domain FID (1024-D with sqrtm, may take ~30s)...")
domain_fid = frechet_distance(real_feats, gen_feats)
logger.info(f"Domain FID (DenseNet-121 features): {domain_fid:.2f}")

# ---- Disease probabilities: sigmoid outputs, expected range 0.08-0.50 ----
# Multi-label sigmoid probs are NOT softmax; values well below 0.5 are correct.
# Dataset positive rate per class is 15-50%, so mean probs ~0.1-0.5 is expected.
logger.info("Running classifier on real and generated images...")
def get_probs(imgs):
    out = []
    for i in range(0, len(imgs), 16):
        batch = imgs[i:i+16].to(device)
        with torch.no_grad():
            out.append(torch.sigmoid(diagnoser.classifier(batch)).cpu())
    return torch.cat(out).numpy()

real_probs = get_probs(real_imgs)
gen_probs  = get_probs(gen_imgs)
real_mean  = real_probs.mean(0)
gen_mean   = gen_probs.mean(0)

print("="*72)
print(f"  DIFFUSION MODEL FIDELITY  (N={FID_N} generated vs real CheXpert)")
print("="*72)
print(f"  Domain FID (DenseNet-121): {domain_fid:.2f}  [lower=better; 0=identical]")
print(f"  Note: FID is always >=0. Negative values indicate a bug in matrix sqrt.")
print()
print(f"  Mean sigmoid probability per class:")
print(f"  Note: values 0.1-0.5 are correct for multi-label sigmoid (not softmax).")
print(f"  {'Class':<22} Real    Generated   |delta|")
for j, cls in enumerate(cfg.LABEL_COLS):
    delta = abs(real_mean[j] - gen_mean[j])
    flag  = " <-- large gap" if delta > 0.10 else ""
    print(f"  {cls:<22} {real_mean[j]:.3f}   {gen_mean[j]:.3f}       {delta:.3f}{flag}")
print("="*72)

# -- Figure (10" wide, single column) --
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

ax = axes[0]
x_pos = np.arange(len(cfg.LABEL_COLS)); width = 0.35
b1 = ax.bar(x_pos-width/2, real_mean, width, label="Real CheXpert", color="#3498DB", alpha=0.85)
b2 = ax.bar(x_pos+width/2, gen_mean,  width, label="Generated",     color="#E74C3C", alpha=0.85)
for bars in [b1, b2]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x()+bar.get_width()/2, h+0.005, f"{h:.2f}",
                ha="center", va="bottom", fontsize=8, color="black")
ax.set_xticks(x_pos); ax.set_xticklabels(cfg.LABEL_COLS, rotation=30, ha="right")
ax.set_ylim(0, 0.75); ax.set_ylabel("Mean predicted probability")
ax.set_title("Disease Prediction: Real vs Generated")
ax.legend(fontsize=11); ax.grid(True, alpha=0.2, axis="y")

ax = axes[1]
ax.bar(["Domain FID\n(DenseNet-121)"], [domain_fid], color="#9B59B6", alpha=0.85, width=0.4)
ax.text(0, domain_fid * 1.05, f"{domain_fid:.1f}", ha="center", va="bottom",
        fontsize=14, color="black", fontweight="bold")
ax.set_ylabel("Frechet Distance (lower=better)")
ax.set_title("Domain FID")
ax.set_ylim(0, max(domain_fid * 1.3, 10)); ax.grid(True, alpha=0.2, axis="y")

fig.suptitle("Diffusion Model Fidelity: Generated vs Real CheXpert Images",
             fontsize=15, color="black", fontweight="bold")
plt.tight_layout()
fid_path = os.path.join(FID_DIR, "fidelity_fid_disease.png")
plt.savefig(fid_path, dpi=150, bbox_inches="tight"); plt.close()
print(f"  Plot -> {fid_path}")


12:34:21 INFO notebook: Generating 1000 images via DDIM (50 steps)...


12:34:28 INFO notebook:   Generated 16/1000
12:34:35 INFO notebook:   Generated 32/1000
12:34:41 INFO notebook:   Generated 48/1000
12:34:47 INFO notebook:   Generated 64/1000
12:34:54 INFO notebook:   Generated 80/1000
12:35:01 INFO notebook:   Generated 96/1000
12:35:07 INFO notebook:   Generated 112/1000
12:35:13 INFO notebook:   Generated 128/1000
12:35:20 INFO notebook:   Generated 144/1000
12:35:27 INFO notebook:   Generated 160/1000
12:35:34 INFO notebook:   Generated 176/1000
12:35:39 INFO notebook:   Generated 192/1000
12:35:46 INFO notebook:   Generated 208/1000
12:35:54 INFO notebook:   Generated 224/1000
12:36:00 INFO notebook:   Generated 240/1000
12:36:06 INFO notebook:   Generated 256/1000
12:36:13 INFO notebook:   Generated 272/1000
12:36:20 INFO notebook:   Generated 288/1000
12:36:25 INFO notebook:   Generated 304/1000
12:36:31 INFO notebook:   Generated 320/1000
12:36:39 INFO notebook:   Generated 336/1000
12:36:46 INFO notebook:   Generated 352/1000
12:36:51 INFO no

  DIFFUSION MODEL FIDELITY  (N=1000 generated vs real CheXpert)
  Domain FID (DenseNet-121): 35.34  [lower=better; 0=identical]
  Note: FID is always >=0. Negative values indicate a bug in matrix sqrt.

  Mean sigmoid probability per class:
  Note: values 0.1-0.5 are correct for multi-label sigmoid (not softmax).
  Class                  Real    Generated   |delta|
  No Finding             0.144   0.091       0.053
  Atelectasis            0.429   0.410       0.020
  Cardiomegaly           0.230   0.149       0.081
  Consolidation          0.376   0.416       0.040
  Edema                  0.361   0.282       0.079
  Pleural Effusion       0.443   0.411       0.032
  Pneumonia              0.212   0.122       0.090
  Pneumothorax           0.130   0.172       0.042
  Plot -> /home/dawood/lab2_rotaion/counterfactual_diff_uncertainty/results/jbhi_fidelity/fidelity_fid_disease.png


#### Note: HFER (Exp 9 FFT Blur Fix) is Now Permanent

The FFT High-Frequency Energy Ratio blur fix (formerly Exp 9) has been
permanently integrated into `cd_dsd/diagnoser.py` and `cd_dsd/baselines.py`.

- `hf_energy_score()` is computed on every image in `diagnose_batch()`
- Calibration stats (`hfer_mean`, `hfer_std`) computed on clean CheXpert in `_calibrate()`
- `score_cd_dsd()` in `BaselineSuite` includes the HFER signal
- AUROC(blur): 0.26 → **1.00** (HFER completely solves blur detection)

No need to re-run the old Exp 9 cell — the fix is always active.

In [13]:
# HFER is permanently integrated — this cell is no longer needed.
# See cd_dsd/diagnoser.py::hf_energy_score() and CDDSDDiagnoser._calibrate()
# and cd_dsd/baselines.py::BaselineSuite.score_cd_dsd()
print("HFER is permanently in cd_dsd/diagnoser.py. No action needed here.")

HFER is permanently in cd_dsd/diagnoser.py. No action needed here.
